In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

# Read data
df = pd.read_csv("HDI.csv", encoding="ISO-8859-9", sep=';', header=0)
df.columns = df.columns.str.strip()

In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import skew,kurtosis
df=pd.read_csv("HDI.csv",encoding="ISO-8859-9",sep=";")
df.columns=df.columns.str.strip()
num=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
for c in num:
    df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",".",regex=False),errors="coerce")
print("Shape:",df.shape)
print("\nMissing values:\n",df.isna().sum())
print("\nDuplicate countries:",df["Country"].duplicated().sum())
print("\nDuplicate rows:",df.duplicated().sum())
print("\nDescriptive statistics:\n",df[num].describe().T)
stats=pd.DataFrame({"Skewness":df[num].skew(),"Kurtosis":df[num].kurtosis()})
print("\nDistribution:\n",stats)
print("\nCorrelation:\n",df[num].corr(method="spearman").round(3))
print("\nCountry duplicates:\n",df[df["Country"].duplicated(keep=False)].sort_values("Country"))
print("\nOutliers (IQR):")
for c in num:
    q1,q3=df[c].quantile([.25,.75])
    lo,hi=q1-1.5*(q3-q1),q3+1.5*(q3-q1)
    print(c,((df[c]<lo)|(df[c]>hi)).sum())


In [ ]:
df=pd.read_csv("HDI.csv",sep=";",encoding="ISO-8859-9")
df.columns=df.columns.str.strip()
df["Country"]=df["Country"].astype(str).str.strip()
cols=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
for c in cols: df[c]=pd.to_numeric(df[c],errors="coerce")
print("VERİ SETİ BOYUTU:",df.shape)
print("\nEKSİK DEĞERLER:\n",df.isna().sum())
print("\nYİNELENEN ÜLKE:",df["Country"].duplicated().sum())
print("YİNELENEN SATIR:",df.duplicated().sum())
print("\nTANIMLAYICI İSTATİSTİKLER:\n",df[cols].describe().round(3).to_string())
assert df["Country"].is_unique,"Ülke adlarında tekrar var."
assert df[cols].notna().all().all(),"Eksik veya sayısal olmayan değer var."
assert (df[["GNI","CO2","CHE","LE","EYS","MYS","EA"]]>=0).all().all(),"Negatif değer kontrol edilmeli."
print("\nKontroller tamamlandı.")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cols = ["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
COLORS = sns.color_palette("husl", len(cols))

def finish(fig, name):
    fig.savefig(name, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# 1. DATA QUALITY
print("="*60, "\nDATA QUALITY\n", "="*60)
print("DATA SHAPE:", df.shape)
missing = pd.DataFrame({"Missing_N":df[cols].isna().sum(),
                        "Missing_%":df[cols].isna().mean()*100})
print("\nMISSING VALUES:\n", missing.round(2))
print("\nUNIQUE COUNTS:\n", df[cols].nunique())

# 2. DESCRIPTIVE STATISTICS
desc = df[cols].describe().T
dist = pd.DataFrame({"Skewness":df[cols].skew(),
                     "Kurtosis":df[cols].kurtosis()})
print("\nDESCRIPTIVE STATISTICS:\n", desc.round(3))
print("\nDISTRIBUTION STATISTICS:\n", dist.round(3))

# 3. PEARSON CORRELATION
corr = df[cols].corr(method="pearson")
print("\nPEARSON CORRELATION:\n", corr.round(3))

# 4. CORRELATION HEATMAP
fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(corr, cmap="RdYlBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j,i,f"{corr.iloc[i,j]:.2f}",ha="center",va="center",fontsize=9)
ax.set_title("Pearson Correlation Matrix", fontweight="bold")
fig.colorbar(im, ax=ax, label="Pearson Correlation")
fig.tight_layout()
finish(fig, "01_correlation_matrix.png")

# 5. VARIABLE DISTRIBUTIONS
fig, axs = plt.subplots(2,4,figsize=(15,8))
for i,c in enumerate(cols):
    axs.flat[i].hist(df[c].dropna(), bins=18, color=COLORS[i], edgecolor="black")
    axs.flat[i].set(title=c, xlabel="Value", ylabel="Frequency")
fig.suptitle("Distribution of Study Variables", fontweight="bold")
fig.tight_layout()
finish(fig, "02_variable_distributions.png")

# 6. BOXPLOTS
fig, axs = plt.subplots(2,4,figsize=(15,8))
for i,c in enumerate(cols):
    axs.flat[i].boxplot(df[c].dropna(), patch_artist=True,
                        boxprops=dict(facecolor=COLORS[i]),
                        medianprops=dict(color="black", linewidth=2))
    axs.flat[i].set(title=c, ylabel="Value")
fig.suptitle("Boxplots of Study Variables", fontweight="bold")
fig.tight_layout()
finish(fig, "03_variable_boxplots.png")

# 7. EXPORT TABLES
with pd.ExcelWriter("DAGA_ML_Preliminary_Analysis.xlsx") as writer:
    desc.to_excel(writer, sheet_name="Descriptive_Statistics")
    dist.to_excel(writer, sheet_name="Distribution_Statistics")
    corr.to_excel(writer, sheet_name="Pearson_Correlation")
    missing.to_excel(writer, sheet_name="Missing_Values")

print("\nANALYSIS COMPLETED SUCCESSFULLY.")
print("Figures: 01–03")
print("Excel: DAGA_ML_Preliminary_Analysis.xlsx")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cols=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
df[cols]=df[cols].apply(pd.to_numeric,errors="coerce")
COLORS=sns.color_palette("husl",n_colors=len(cols))

# 1. DATA QUALITY
print("="*70)
print("DATA QUALITY")
print("="*70)
print("DATA SHAPE:",df.shape)
print("\nMISSING VALUES:")
print(pd.DataFrame({"Missing_N":df[cols].isna().sum(),"Missing_%":(df[cols].isna().mean()*100).round(2)}).to_string())
print("\nUNIQUE COUNTS:")
print(df[cols].nunique().to_string())

# 2. DESCRIPTIVE STATISTICS
desc=df[cols].describe().T
dist=pd.DataFrame({"Skewness":df[cols].skew(),"Kurtosis":df[cols].kurtosis()})
print("\nDESCRIPTIVE STATISTICS:")
print(desc.round(3).to_string())
print("\nDISTRIBUTION STATISTICS:")
print(dist.round(3).to_string())

# 3. PEARSON CORRELATION
corr=df[cols].corr(method="pearson")
print("\nPEARSON CORRELATION MATRIX:")
print(corr.round(3).to_string())

# 4. PEARSON CORRELATION HEATMAP
fig,ax=plt.subplots(figsize=(11,9))
im=ax.imshow(corr,cmap="RdYlBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols,rotation=45,ha="right",fontweight="bold")
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols,fontweight="bold")
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j,i,f"{corr.iloc[i,j]:.3f}",ha="center",va="center",color="black",fontweight="bold",fontsize=10)
ax.set_title("Pearson Correlation Matrix",fontsize=16,fontweight="bold",color="black",pad=15)
fig.colorbar(im,ax=ax,label="Pearson Correlation")
fig.tight_layout()
finish(fig,"01_correlation_matrix.png")

# 5. VARIABLE DISTRIBUTIONS
fig,axs=plt.subplots(2,4,figsize=(17,9))
for i,c in enumerate(cols):
    ax=axs.flat[i]
    values=df[c].dropna()
    ax.hist(values,bins=18,color=COLORS[i],edgecolor="black",alpha=.9)
    ax.set_title(c,fontweight="bold",color="black",fontsize=12)
    ax.set_xlabel("Value",fontweight="bold")
    ax.set_ylabel("Frequency",fontweight="bold")
    ax.tick_params(labelcolor="black")
    ax.grid(axis="y",alpha=.2)
fig.suptitle("Distribution of Study Variables",fontsize=16,fontweight="bold",color="black")
fig.tight_layout()
finish(fig,"02_variable_distributions.png")

# 6. BOXPLOTS
fig,axs=plt.subplots(2,4,figsize=(17,9))
for i,c in enumerate(cols):
    ax=axs.flat[i]
    ax.boxplot(df[c].dropna(),patch_artist=True,boxprops=dict(facecolor=COLORS[i],alpha=.8),medianprops=dict(color="black",linewidth=2))
    ax.set_title(c,fontweight="bold",color="black")
    ax.set_ylabel("Value",fontweight="bold")
    ax.grid(axis="y",alpha=.2)
fig.suptitle("Boxplots of Study Variables",fontsize=16,fontweight="bold",color="black")
fig.tight_layout()
finish(fig,"03_variable_boxplots.png")

# 7. SAVE STATISTICAL TABLES
with pd.ExcelWriter("DAGA_ML_Preliminary_Analysis.xlsx") as writer:
    desc.to_excel(writer,sheet_name="Descriptive_Statistics")
    dist.to_excel(writer,sheet_name="Distribution_Statistics")
    corr.to_excel(writer,sheet_name="Pearson_Correlation")
    missing=pd.DataFrame({"Missing_N":df[cols].isna().sum(),"Missing_%":df[cols].isna().mean()*100})
    missing.to_excel(writer,sheet_name="Missing_Values")

print("\nANALYSIS COMPLETED SUCCESSFULLY.")
print("Figures: 01–03")
print("Excel: DAGA_ML_Preliminary_Analysis.xlsx")

In [ ]:
def iqr_outliers(s):
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    return ((s<q1-1.5*iqr)|(s>q3+1.5*iqr))
out=pd.DataFrame({c:iqr_outliers(df[c]).sum() for c in cols},index=["IQR outlier count"]).T
print("IQR AYKIRI DEĞER SAYILARI:\n",out.to_string())
print("\nGNI ve CO2 dönüşüm öncesi/sonrası çarpıklık:")
for c,z in [("GNI",np.log(df["GNI"])),("CO2",np.log1p(df["CO2"]))]:
    print(f"{c}: {df[c].skew():.3f} -> {z.skew():.3f}")
fig,axs=plt.subplots(1,2,figsize=(12,5))
for ax,c,z,title,color in [(axs[0],"GNI",np.log(df["GNI"]),"GNI: Original vs Log",COLORS[0]),(axs[1],"CO2",np.log1p(df["CO2"]),"CO2: Original vs log1p",COLORS[1])]:
    ax.boxplot([df[c],z],tick_labels=["Original","Transformed"],patch_artist=True,boxprops=dict(facecolor=color,alpha=.7),medianprops=dict(color="black",linewidth=2))
    ax.set_title(title,color="black",fontweight="bold"); ax.set_ylabel("Value")
finish(fig,"03_transformation_comparison.png")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]

Xdf=df[features].copy()
Xdf["GNI"]=np.log(Xdf["GNI"])
Xdf["CO2"]=np.log1p(Xdf["CO2"])

scaler=StandardScaler()
X=pd.DataFrame(scaler.fit_transform(Xdf),columns=features,index=df.index)

print("ANA MODEL DEĞİŞKENLERİ:",features)
print("\nSTANDARDİZE EDİLMİŞ VERİ ÖZETİ:\n",X.describe().round(3).to_string())
print("\nOrtalama değerler 0'a, standart sapmalar 1'e yakın olmalıdır.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

COLORS=plt.cm.tab10(np.linspace(0,1,7))

def finish(fig,name):
    fig.savefig(name,dpi=600,bbox_inches="tight")
    plt.show()
    plt.close(fig)

X=np.asarray(X,dtype=float)
features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]

pca_full=PCA().fit(X)
evr=pca_full.explained_variance_ratio_
cum=np.cumsum(evr)

pca_table=pd.DataFrame({
    "Component":[f"PC{i+1}" for i in range(len(evr))],
    "Eigenvalue":pca_full.explained_variance_,
    "Explained variance":evr,
    "Cumulative variance":cum
})

print("PCA SONUÇLARI:\n",pca_table.round(4).to_string(index=False))

if cum[-1]<.90:
    raise ValueError("%90 varyans eşiğine ulaşılamadı.")

n_pc=np.argmax(cum>=.90)+1
pca=PCA(n_components=n_pc)
Z=pca.fit_transform(X)

loadings=pd.DataFrame(
    pca.components_.T,index=features,
    columns=[f"PC{i+1}" for i in range(n_pc)]
)

print(f"\n%90 varyans için bileşen sayısı: {n_pc}")
print("\nPCA YÜKLERİ:\n",loadings.round(3).to_string())

fig,axs=plt.subplots(1,2,figsize=(13,5))

axs[0].bar(range(1,len(evr)+1),evr*100,
           color=COLORS,edgecolor="black")
axs[0].set(xlabel="Principal Component",
           ylabel="Explained Variance (%)",
           title="PCA Explained Variance")

axs[1].plot(range(1,len(cum)+1),cum*100,
            marker="o",linewidth=2)
axs[1].axhline(90,color="black",linestyle="--")
axs[1].set(xlabel="Number of Components",
           ylabel="Cumulative Variance (%)",
           title="Cumulative Explained Variance")

for ax in axs:
    ax.tick_params(colors="black")

for i,v in enumerate(evr*100):
    axs[0].text(i+1,v+1,f"{v:.1f}",ha="center")

finish(fig,"04_pca_variance.png")

fig,ax=plt.subplots(figsize=(max(7,n_pc*1.5),5))
im=ax.imshow(loadings,cmap="coolwarm",vmin=-1,vmax=1,aspect="auto")
ax.set_xticks(range(n_pc),loadings.columns)
ax.set_yticks(range(len(features)),features)

for i in range(len(features)):
    for j in range(n_pc):
        ax.text(j,i,f"{loadings.iloc[i,j]:.2f}",
                ha="center",va="center",color="black")

ax.set_title("PCA Component Loadings",fontweight="bold")
fig.colorbar(im,ax=ax,label="Component Loading")
finish(fig,"05_pca_loadings.png")

pd.DataFrame(Z,columns=loadings.columns).to_csv("PCA_Scores.csv",index=False)
pca_table.to_csv("PCA_Results.csv",index=False)
loadings.to_csv("PCA_Loadings.csv")

print("\nPCA ANALİZİ TAMAMLANDI.")

In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt,matplotlib as mpl
from sklearn.decomposition import PCA
from adjustText import adjust_text

features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]
if "X" not in globals(): raise ValueError("Önce X veri setini oluşturun.")
X=np.asarray(X,dtype=float)
if X.ndim!=2 or X.shape[1]!=7 or not np.isfinite(X).all():
    raise ValueError("X, 7 sütunlu ve eksiksiz sayısal veri içermelidir.")

mpl.rcParams.update({"font.family":"Times New Roman","font.size":12})
colors=plt.cm.tab10(np.linspace(0,1,len(features)))
pca=PCA().fit(X)
scores=pca.transform(X)
loadings=pca.components_.T
eig=pca.explained_variance_
evr=pca.explained_variance_ratio_
cum=np.cumsum(evr)
n_pc=np.argmax(cum>=.90)+1 if cum[-1]>=.90 else len(evr)
pairs=[(0,1),(0,2),(1,2),(0,3)]
pairs=[p for p in pairs if p[1]<len(evr)]

print("\n=== PCA EXPLAINED VARIANCE ===")
for i,v in enumerate(evr,1): print(f"PC{i}: {v*100:.2f}%")
print(f"Cumulative variance (PC1-PC2): {cum[1]*100:.2f}%")
print(f"Components for 90% variance: {n_pc}")

print("\n=== PCA LOADINGS ===")
print(pd.DataFrame(loadings,index=features,
    columns=[f"PC{i}" for i in range(1,len(evr)+1)]).round(4))

print("\n=== VARIABLE COORDINATES ===")
for i,j in pairs:
    print(f"\nPC{i+1} x PC{j+1}")
    for k,f in enumerate(features):
        x,y=loadings[k,i]*np.sqrt(eig[i]),loadings[k,j]*np.sqrt(eig[j])
        print(f"{f}: ({x:.4f}, {y:.4f})")

fig,axes=plt.subplots(2,2,figsize=(21,17))
for ax,(i,j) in zip(axes.flat,pairs):
    theta=np.linspace(0,2*np.pi,400)
    ax.plot(np.cos(theta),np.sin(theta),"--",color="gray",lw=1)
    ax.axhline(0,color="black",lw=.8)
    ax.axvline(0,color="black",lw=.8)

    sx=scores[:,i]/max(np.max(np.abs(scores[:,i])),1e-12)
    sy=scores[:,j]/max(np.max(np.abs(scores[:,j])),1e-12)
    ax.scatter(sx,sy,s=25,alpha=.22,color="steelblue",zorder=1)

    texts=[]
    for k,f in enumerate(features):
        x=loadings[k,i]*np.sqrt(eig[i])
        y=loadings[k,j]*np.sqrt(eig[j])
        c=colors[k]
        ax.annotate("",xy=(x,y),xytext=(0,0),
            arrowprops=dict(arrowstyle="-|>",color=c,lw=2.2,
                            mutation_scale=14),
            zorder=3)
        texts.append(ax.text(x*1.08,y*1.08,f"{f}\n({loadings[k,i]:.2f}, {loadings[k,j]:.2f})",
            color=c,fontsize=10,fontweight="bold",ha="center",va="center",
            bbox=dict(boxstyle="round,pad=.25",fc="white",ec=c,alpha=.88)))

    adjust_text(texts,ax=ax,only_move={"text":"xy"},
        arrowprops=dict(arrowstyle="-",color="gray",lw=.7),
        force_text=(.8,.8),expand=(1.25,1.4),
        ensure_inside_axes=True)

    ax.set(xlim=(-1.4,1.4),ylim=(-1.4,1.4),
        xlabel=f"PC{i+1} ({evr[i]*100:.2f}%)",
        ylabel=f"PC{j+1} ({evr[j]*100:.2f}%)",
        title=f"Correlation Circle: PC{i+1} vs PC{j+1}")
    ax.set_aspect("equal")
    ax.grid(True,ls="--",alpha=.25)
    ax.set_axisbelow(True)

for ax in axes.flat[len(pairs):]: ax.set_visible(False)

fig.suptitle("PCA Correlation Circles: Country Scores and Variable Loadings",
    fontsize=18,fontweight="bold")
fig.tight_layout(rect=[0,0,1,.96])
fig.savefig("PCA_Correlation_Circles.png",dpi=600,bbox_inches="tight")
fig.savefig("PCA_Correlation_Circles.pdf",bbox_inches="tight")
plt.show()

pd.DataFrame(scores,columns=[f"PC{i}" for i in range(1,len(evr)+1)]).to_csv("PCA_Scores.csv",index=False)
pd.DataFrame({"Component":[f"PC{i}" for i in range(1,len(evr)+1)],
    "Eigenvalue":eig,"Explained variance":evr,
    "Cumulative variance":cum}).to_csv("PCA_Results.csv",index=False)
pd.DataFrame(loadings,index=features,
    columns=[f"PC{i}" for i in range(1,len(evr)+1)]).to_csv("PCA_Loadings.csv")

print(f"\nObservations used: {len(X)}")
print("PCA ANALYSIS COMPLETED.")
print("Figures: PCA_Correlation_Circles.png / PDF")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]

Xdf=df[features].apply(pd.to_numeric,errors="coerce").copy()
Xdf["GNI"]=np.log(Xdf["GNI"].clip(lower=1))
Xdf["CO2"]=np.log1p(Xdf["CO2"].clip(lower=0))
Xdf=Xdf.replace([np.inf,-np.inf],np.nan)

if Xdf.isna().any().any():
    raise ValueError("Veri setinde eksik veya geçersiz değer var.")

X=pd.DataFrame(StandardScaler().fit_transform(Xdf),
               columns=features,index=df.index)

print("ANA MODEL DEĞİŞKENLERİ:",features)
print("\nSTANDARDİZE EDİLMİŞ VERİ ÖZETİ:\n",X.describe().round(3))

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"]="4"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

K_RANGE=range(2,8)
SEED=42

if "Z" not in globals():
    raise ValueError("Önce PCA analizini çalıştırarak Z değişkenini oluşturun.")

Z=np.asarray(Z,dtype=float)

def finish(fig,name):
    fig.savefig(name,dpi=600,bbox_inches="tight")
    plt.show()
    plt.close(fig)

gmm_results=[]

for k in K_RANGE:
    model=GaussianMixture(
        n_components=k,covariance_type="full",n_init=10,
        reg_covar=1e-5,random_state=SEED,max_iter=500
    )
    model.fit(Z)
    labels=model.predict(Z)

    gmm_results.append({
        "K":k,
        "AIC":model.aic(Z),
        "BIC":model.bic(Z),
        "Silhouette":silhouette_score(Z,labels),
        "Davies-Bouldin":davies_bouldin_score(Z,labels)
    })

gmm_table=pd.DataFrame(gmm_results)
print("GMM MODEL SEÇİMİ:\n",gmm_table.round(4).to_string(index=False))

best_aic=int(gmm_table.loc[gmm_table["AIC"].idxmin(),"K"])
best_bic=int(gmm_table.loc[gmm_table["BIC"].idxmin(),"K"])
best_sil=int(gmm_table.loc[gmm_table["Silhouette"].idxmax(),"K"])
best_db=int(gmm_table.loc[gmm_table["Davies-Bouldin"].idxmin(),"K"])

print("\nOPTİMAL K DEĞERLERİ")
print(f"AIC minimum: K={best_aic}")
print(f"BIC minimum: K={best_bic}")
print(f"Silhouette maximum: K={best_sil}")
print(f"Davies-Bouldin minimum: K={best_db}")

metrics=[
    ("AIC","AIC (Lower is Better)"),
    ("BIC","BIC (Lower is Better)"),
    ("Silhouette","Silhouette (Higher is Better)"),
    ("Davies-Bouldin","Davies-Bouldin (Lower is Better)")
]

fig,axs=plt.subplots(2,2,figsize=(13,9))
colors=sns.color_palette("husl",4)

for ax,(col,title),color in zip(axs.flat,metrics,colors):
    ax.plot(gmm_table["K"],gmm_table[col],marker="o",
            linewidth=2.5,markersize=7,color=color)
    ax.set_xticks(list(K_RANGE))
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel(col)
    ax.set_title(title,fontweight="bold")
    ax.grid(alpha=.25)

    for x,y in zip(gmm_table["K"],gmm_table[col]):
        ax.annotate(f"{y:.3f}",(x,y),xytext=(0,8),
                    textcoords="offset points",ha="center",fontsize=9)

fig.suptitle("GMM Cluster Number Selection",fontsize=15,fontweight="bold")
fig.tight_layout()
finish(fig,"06_gmm_model_selection.png")

gmm_table.to_csv("GMM_Model_Selection.csv",index=False)

print("\nGMM MODEL SEÇİMİ TAMAMLANDI.")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# SETTINGS
K_FINAL=3
B_BOOT=500
SEED=42
rng=np.random.default_rng(SEED)
Z_array=np.asarray(Z)
n=len(Z_array)

# 1. FINAL GMM MODEL
gmm_final=GaussianMixture(
    n_components=K_FINAL,
    covariance_type="full",
    n_init=20,
    reg_covar=1e-5,
    max_iter=500,
    random_state=SEED
)
gmm_final.fit(Z_array)

labels=gmm_final.predict(Z_array)
probabilities=gmm_final.predict_proba(Z_array)
max_probability=probabilities.max(axis=1)

# 2. CLUSTER SIZES AND PROPORTIONS
cluster_counts=pd.Series(labels).value_counts().sort_index()
cluster_summary=pd.DataFrame({
    "Cluster":range(K_FINAL),
    "N":[cluster_counts.get(i,0) for i in range(K_FINAL)],
    "Percentage":[100*cluster_counts.get(i,0)/n for i in range(K_FINAL)]
})

print("\nCLUSTER SIZE AND PROPORTION")
print(cluster_summary.round(3).to_string(index=False))

# 3. POSTERIOR PROBABILITY SUMMARY
posterior_summary=pd.DataFrame({
    "Cluster":range(K_FINAL),
    "Mean_Max_Probability":[max_probability[labels==i].mean() for i in range(K_FINAL)],
    "Median_Max_Probability":[np.median(max_probability[labels==i]) for i in range(K_FINAL)],
    "Min_Max_Probability":[max_probability[labels==i].min() for i in range(K_FINAL)],
    "Below_0.70_N":[np.sum((labels==i)&(max_probability<0.70)) for i in range(K_FINAL)],
    "Below_0.70_%":[100*np.mean(max_probability[(labels==i)]<0.70) for i in range(K_FINAL)]
})

print("\nPOSTERIOR PROBABILITY SUMMARY")
print(posterior_summary.round(4).to_string(index=False))

print("\nOVERALL POSTERIOR PROBABILITY")
print(pd.Series(max_probability).describe().round(4).to_string())

# 4. BOOTSTRAP STABILITY: ARI AND NMI
ari_scores=[]
nmi_scores=[]
failed=0

for b in range(B_BOOT):
    idx=rng.choice(n,size=n,replace=True)
    Z_boot=Z_array[idx]

    try:
        model=GaussianMixture(
            n_components=K_FINAL,
            covariance_type="full",
            n_init=5,
            reg_covar=1e-5,
            max_iter=500,
            random_state=SEED+b+1
        )
        model.fit(Z_boot)
        boot_labels=model.predict(Z_array)

        ari_scores.append(adjusted_rand_score(labels,boot_labels))
        nmi_scores.append(normalized_mutual_info_score(labels,boot_labels))
    except Exception:
        failed+=1

ari_scores=np.asarray(ari_scores)
nmi_scores=np.asarray(nmi_scores)

bootstrap_summary=pd.DataFrame({
    "Metric":["ARI","NMI"],
    "Mean":[ari_scores.mean(),nmi_scores.mean()],
    "SD":[ari_scores.std(ddof=1),nmi_scores.std(ddof=1)],
    "Median":[np.median(ari_scores),np.median(nmi_scores)],
    "2.5%":[np.quantile(ari_scores,.025),np.quantile(nmi_scores,.025)],
    "97.5%":[np.quantile(ari_scores,.975),np.quantile(nmi_scores,.975)]
})

print("\nBOOTSTRAP STABILITY RESULTS")
print(bootstrap_summary.round(4).to_string(index=False))
print(f"\nSuccessful bootstrap repetitions: {len(ari_scores)}/{B_BOOT}")
print(f"Failed repetitions: {failed}")

# 5. BOOTSTRAP DISTRIBUTION PLOT
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for ax,values,title,color in zip(
    axs,
    [ari_scores,nmi_scores],
    ["Bootstrap ARI","Bootstrap NMI"],
    sns.color_palette("husl",2)
):
    ax.hist(values,bins=25,color=color,edgecolor="black",alpha=.85)
    ax.axvline(np.mean(values),color="black",linestyle="--",linewidth=2,label=f"Mean={np.mean(values):.3f}")
    ax.set_title(title,fontweight="bold",color="black")
    ax.set_xlabel("Stability Score")
    ax.set_ylabel("Frequency")
    ax.legend()
    ax.grid(axis="y",alpha=.2)

fig.suptitle("Bootstrap Stability of GMM Clustering (K=3)",fontsize=14,fontweight="bold")
fig.tight_layout()
finish(fig,"07_gmm_bootstrap_stability.png")

# 6. POSTERIOR PROBABILITY DISTRIBUTION
fig,ax=plt.subplots(figsize=(8,5))
for i,color in zip(range(K_FINAL),sns.color_palette("husl",K_FINAL)):
    ax.hist(max_probability[labels==i],bins=20,alpha=.55,label=f"Cluster {i+1}",color=color,edgecolor="black")
ax.axvline(.70,color="black",linestyle="--",linewidth=1.5,label="0.70 threshold")
ax.set_title("Maximum Posterior Probability by Cluster",fontweight="bold")
ax.set_xlabel("Maximum Posterior Probability")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
finish(fig,"08_gmm_posterior_probabilities.png")

# 7. SAVE RESULTS
posterior_df=pd.DataFrame(probabilities,columns=[f"Posterior_Cluster_{i+1}" for i in range(K_FINAL)])
posterior_df.insert(0,"Assigned_Cluster",labels+1)
posterior_df["Maximum_Posterior"]=max_probability

with pd.ExcelWriter("GMM_K3_Validation_Results.xlsx") as writer:
    cluster_summary.to_excel(writer,sheet_name="Cluster_Sizes",index=False)
    posterior_summary.to_excel(writer,sheet_name="Posterior_Summary",index=False)
    posterior_df.to_excel(writer,sheet_name="Observation_Posteriors",index=False)
    bootstrap_summary.to_excel(writer,sheet_name="Bootstrap_Stability",index=False)
    pd.DataFrame({"ARI":ari_scores,"NMI":nmi_scores}).to_excel(writer,sheet_name="Bootstrap_Replications",index=False)

print("\nALL VALIDATION ANALYSES COMPLETED.")


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# SETTINGS
K_VALUES=[2,3,5]
SEED=42
cols=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
sns.set_theme(style="white",font="Arial")

# DATA PREPARATION
X_original=df[cols].dropna().copy()
Z_array=np.asarray(Z)

if len(X_original)!=len(Z_array):
    raise ValueError(f"Gözlem sayıları eşleşmiyor: X={len(X_original)}, Z={len(Z_array)}.")
if not np.isfinite(Z_array).all():
    raise ValueError("Z içerisinde NaN veya sonsuz değer bulunuyor.")

X_original=X_original.reset_index(drop=True)
X_scaled=pd.DataFrame(StandardScaler().fit_transform(X_original),columns=cols)

# MODEL ESTIMATION
size_results=[]
raw_profiles={}
z_profiles={}
profile_data={}

for k in K_VALUES:
    model=GaussianMixture(n_components=k,covariance_type="full",n_init=20,reg_covar=1e-5,max_iter=500,random_state=SEED)
    model.fit(Z_array)
    labels=model.predict(Z_array)
    raw_data=X_original.copy()
    raw_data["Cluster"]=labels+1
    scaled_data=X_scaled.copy()
    scaled_data["Cluster"]=labels+1
    counts=pd.Series(labels+1).value_counts().sort_index()

    for c in range(1,k+1):
        size_results.append({"K":k,"Cluster":c,"N":int(counts.get(c,0)),"Percentage":100*counts.get(c,0)/len(labels)})

    raw_mean=raw_data.groupby("Cluster")[cols].mean()
    z_mean=scaled_data.groupby("Cluster")[cols].mean()
    raw_profiles[k]=raw_mean
    z_profiles[k]=z_mean
    profile_data[k]={"labels":labels,"model":model,"raw":raw_mean,"z":z_mean}

# PRINT TABLES
size_table=pd.DataFrame(size_results)
print("\nCLUSTER SIZES AND PROPORTIONS")
print(size_table.round(3).to_string(index=False))

for k in K_VALUES:
    print(f"\n{'='*65}\nK={k}: ORIGINAL VARIABLE MEANS\n{'='*65}")
    print(raw_profiles[k].round(3).to_string())
    print(f"\n{'='*65}\nK={k}: STANDARDIZED CLUSTER PROFILES\n{'='*65}")
    print(z_profiles[k].round(3).to_string())

# 1. CLUSTER SIZE BAR CHARTS WITH LABELS
fig,axs=plt.subplots(1,3,figsize=(16,5))
for ax,k in zip(axs,K_VALUES):
    d=size_table[size_table["K"]==k]
    bars=ax.bar(d["Cluster"].astype(str),d["Percentage"],color=sns.color_palette("husl",k),edgecolor="black",linewidth=1)
    for bar,(_,row) in zip(bars,d.iterrows()):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f'N={int(row["N"])}\n{row["Percentage"]:.1f}%',ha="center",va="bottom",fontsize=9,fontweight="bold",color="black")
    ax.set_title(f"Cluster Sizes (K={k})",fontweight="bold")
    ax.set_xlabel("Cluster",fontweight="bold")
    ax.set_ylabel("Observations (%)",fontweight="bold")
    ax.set_ylim(0,max(d["Percentage"])*1.22)
    ax.grid(axis="y",alpha=.2)
fig.tight_layout()
finish(fig,"09_gmm_cluster_sizes.png")

# 2. STANDARDIZED PROFILE HEATMAPS WITH NUMERIC LABELS
fig,axs=plt.subplots(1,3,figsize=(20,6))
for ax,k in zip(axs,K_VALUES):
    sns.heatmap(z_profiles[k],annot=True,fmt=".2f",cmap="RdBu_r",center=0,linewidths=.8,linecolor="white",annot_kws={"size":10,"weight":"bold","color":"black"},cbar_kws={"label":"Standardized Mean"},ax=ax)
    ax.set_title(f"GMM Cluster Profiles (K={k})",fontweight="bold",fontsize=13)
    ax.set_xlabel("Variables",fontweight="bold")
    ax.set_ylabel("Cluster",fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontweight="bold")
    ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontweight="bold")
fig.tight_layout()
finish(fig,"10_gmm_profile_heatmaps.png")

# 3. STANDARDIZED PROFILE LINE PLOTS WITH POINT LABELS
fig,axs=plt.subplots(1,3,figsize=(20,6))
for ax,k in zip(axs,K_VALUES):
    data=z_profiles[k]
    colors=sns.color_palette("husl",k)
    x=np.arange(len(cols))
    for i,(cluster,row) in enumerate(data.iterrows()):
        y=row.values
        ax.plot(x,y,marker="o",linewidth=2.3,markersize=7,color=colors[i],label=f"Cluster {cluster}")
        for xi,yi in zip(x,y):
            ax.annotate(f"{yi:.2f}",(xi,yi),xytext=(0,9 if yi>=0 else -14),textcoords="offset points",ha="center",fontsize=8,fontweight="bold",color="black")
    ax.axhline(0,color="black",linewidth=1,linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(cols,rotation=45,ha="right",fontweight="bold")
    ax.set_title(f"Standardized Cluster Profiles (K={k})",fontweight="bold",fontsize=12)
    ax.set_xlabel("Variables",fontweight="bold")
    ax.set_ylabel("Standardized Mean",fontweight="bold")
    ax.legend(title="Cluster",fontsize=9)
    ax.grid(axis="y",alpha=.2)
fig.tight_layout()
finish(fig,"11_gmm_profile_lineplots.png")

# 4. ORIGINAL VARIABLE MEAN HEATMAPS WITH LABELS
fig,axs=plt.subplots(1,3,figsize=(21,6))
for ax,k in zip(axs,K_VALUES):
    sns.heatmap(raw_profiles[k],annot=True,fmt=".2f",cmap="YlGnBu",linewidths=.8,linecolor="white",annot_kws={"size":9,"weight":"bold","color":"black"},cbar_kws={"label":"Original Mean"},ax=ax)
    ax.set_title(f"Original Variable Means (K={k})",fontweight="bold",fontsize=12)
    ax.set_xlabel("Variables",fontweight="bold")
    ax.set_ylabel("Cluster",fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontweight="bold")
    ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontweight="bold")
fig.tight_layout()
finish(fig,"12_gmm_raw_mean_heatmaps.png")

# 5. EXPORT TABLES
with pd.ExcelWriter("GMM_K2_K3_K5_Profile_Comparison.xlsx") as writer:
    size_table.to_excel(writer,sheet_name="Cluster_Sizes",index=False)
    for k in K_VALUES:
        raw_profiles[k].to_excel(writer,sheet_name=f"K{k}_Raw")
        z_profiles[k].to_excel(writer,sheet_name=f"K{k}_Standardized")

print("\nCOMPARISON COMPLETED SUCCESSFULLY.")
print("Figures: 09, 10, 11, 12")
print("Excel: GMM_K2_K3_K5_Profile_Comparison.xlsx")
print("Z shape:",Z_array.shape)
print("Original data shape:",X_original.shape)
print("GMM observations:",len(Z_array))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

SEED=42
sns.set_theme(style="white",font="Arial")

# DATA VALIDATION
Z_array=np.asarray(Z)
if Z_array.ndim!=2 or len(df)!=len(Z_array): raise ValueError("df ve Z boyutları uyumsuz.")
if not np.isfinite(Z_array).all(): raise ValueError("Z içerisinde NaN veya sonsuz değer var.")
if "Country" not in df.columns: raise ValueError("Country sütunu bulunamadı.")
profile_features=list(dict.fromkeys(features+["HDI"]))
if not set(profile_features).issubset(df.columns): raise ValueError("Profil değişkenleri df içerisinde bulunamadı.")
if df[profile_features].isna().any().any(): raise ValueError("Profil değişkenlerinde eksik değer var.")
if len(gmm_table)==0 or not {"K","BIC"}.issubset(gmm_table.columns): raise ValueError("gmm_table K ve BIC sütunlarını içermeli.")

# BIC MODEL SELECTION
best_k=int(gmm_table.loc[gmm_table["BIC"].idxmin(),"K"])
gmm=GaussianMixture(n_components=best_k,covariance_type="full",n_init=20,reg_covar=1e-5,max_iter=500,random_state=SEED).fit(Z_array)
labels=gmm.predict(Z_array)
probs=gmm.predict_proba(Z_array)

# REGIME ASSIGNMENT
df=df.copy()
df["GMM_Regime"]=labels+1
df["GMM_Confidence"]=probs.max(axis=1)
df["GMM_Entropy"]=-np.sum(probs*np.log(np.maximum(probs,1e-12)),axis=1)/np.log(best_k) if best_k>1 else 0

# PROFILES
raw_profile=df.groupby("GMM_Regime")[profile_features].mean()
scaled=pd.DataFrame(StandardScaler().fit_transform(df[profile_features]),columns=profile_features,index=df.index)
scaled["GMM_Regime"]=df["GMM_Regime"]
std_profile=scaled.groupby("GMM_Regime")[profile_features].mean()
sizes=df["GMM_Regime"].value_counts().sort_index()
size_table=pd.DataFrame({"N":sizes,"Percentage":sizes/len(df)*100})

# MEMBERSHIP PROBABILITIES
prob_table=pd.DataFrame(probs,columns=[f"Regime_{i+1}" for i in range(best_k)])
prob_table.insert(0,"Country",df["Country"].values)
prob_table.insert(1,"GMM_Regime",df["GMM_Regime"].values)
prob_table.insert(2,"Confidence",df["GMM_Confidence"].values)
prob_table.insert(3,"Entropy",df["GMM_Entropy"].values)

# RESULTS
print(f"SELECTED GMM COMPONENTS: {best_k}")
print("\nCLUSTER SIZES:\n",size_table.round(2).to_string())
print("\nRAW REGIME PROFILES:\n",raw_profile.round(3).to_string())
print("\nSTANDARDIZED REGIME PROFILES:\n",std_profile.round(3).to_string())
print("\nMEAN MEMBERSHIP CONFIDENCE:",round(df["GMM_Confidence"].mean(),4))
print("MEAN NORMALIZED ENTROPY:",round(df["GMM_Entropy"].mean(),4))
print("\nMEMBERSHIP PROBABILITIES (FIRST 10):\n",prob_table.head(10).round(3).to_string(index=False))

# FIGURES
fig,axs=plt.subplots(1,2,figsize=(15,6))
bars=axs[0].bar([f"Regime {i}" for i in sizes.index],sizes.values,color=COLORS[:best_k],edgecolor="black")
axs[0].set_title(f"GMM Regime Sizes (K={best_k})",fontweight="bold")
axs[0].set_ylabel("Number of observations",fontweight="bold")
for bar,n,p in zip(bars,sizes.values,size_table["Percentage"].values):
    axs[0].text(bar.get_x()+bar.get_width()/2,n+1,f"{n}\n({p:.1f}%)",ha="center",fontweight="bold",color="black")
axs[0].set_ylim(0,max(sizes.values)*1.18)
axs[0].grid(axis="y",alpha=.2)
im=axs[1].imshow(std_profile.T,cmap="RdYlBu_r",aspect="auto",vmin=-2,vmax=2)
axs[1].set_xticks(range(best_k),[f"Regime {i}" for i in range(1,best_k+1)],fontweight="bold")
axs[1].set_yticks(range(len(profile_features)),profile_features,fontweight="bold")
for i in range(len(profile_features)):
    for j in range(best_k):
        axs[1].text(j,i,f"{std_profile.iloc[j,i]:.2f}",ha="center",va="center",color="black",fontweight="bold",fontsize=9)
axs[1].set_title("Standardized Regime Profiles",fontweight="bold")
fig.colorbar(im,ax=axs[1],shrink=.8,label="Standardized mean")
fig.tight_layout()
finish(fig,"07_gmm_regime_profiles.png")

# EXCEL EXPORT
with pd.ExcelWriter("GMM_Regime_Results.xlsx") as writer:
    df.to_excel(writer,sheet_name="Country_Regimes",index=False)
    size_table.to_excel(writer,sheet_name="Regime_Sizes")
    raw_profile.to_excel(writer,sheet_name="Raw_Profiles")
    std_profile.to_excel(writer,sheet_name="Standardized_Profiles")
    prob_table.to_excel(writer,sheet_name="Membership_Probabilities",index=False)

print("\nANALYSIS COMPLETED. SELECTED K =",best_k)

In [ ]:
groups=[df.loc[df["GMM_Regime"]==k,"HDI"].values for k in sorted(df["GMM_Regime"].unique())]
H,p=kruskal(*groups)
print(f"Kruskal-Wallis H={H:.4f}, p-value={p:.6g}")
hdi_profile=df.groupby("GMM_Regime")["HDI"].agg(["count","mean","std","median","min","max"])
print("\nREJİMLERE GÖRE HDI:\n",hdi_profile.round(4).to_string())
fig,ax=plt.subplots(figsize=(9,5))
data=[df.loc[df["GMM_Regime"]==k,"HDI"].values for k in sorted(df["GMM_Regime"].unique())]
bp=ax.boxplot(data,patch_artist=True,showmeans=True,meanprops=dict(marker="D",markerfacecolor="white",markeredgecolor="black"))
for patch,color in zip(bp["boxes"],COLORS): patch.set_facecolor(color); patch.set_alpha(.8)
ax.set_xticks(range(1,best_k+1),[f"Regime {i}" for i in range(1,best_k+1)],fontweight="bold")
ax.set_ylabel("Human Development Index"); ax.set_title(f"HDI Across GMM Regimes (Kruskal-Wallis p={p:.3g})",color="black")
finish(fig,"08_hdi_by_regime.png")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# SETTINGS
sns.set_theme(style="white",font="Arial")
COLORS=["#00B4D8","#FF6B35","#7B2CBF","#2DC653","#F72585"]
SEED=42
groups=sorted(df["GMM_Regime"].dropna().unique())
data=df[["GMM_Regime","HDI"]].dropna().copy()
values=[data.loc[data["GMM_Regime"]==k,"HDI"].values for k in groups]

# KRUSKAL-WALLIS AND EFFECT SIZE
H,p=kruskal(*values)
n=len(data)
k=len(groups)
epsilon_sq=max(0,(H-k+1)/(n-k))
profile=data.groupby("GMM_Regime")["HDI"].agg(["count","mean","std","median","min","max"])
print(f"Kruskal-Wallis H={H:.4f}, p={p:.6e}, epsilon-squared={epsilon_sq:.4f}")
print("\nHDI REGIME PROFILES:\n",profile.round(4).to_string())

# DUNN POST-HOC WITH HOLM CORRECTION
dunn=sp.posthoc_dunn(data,val_col="HDI",group_col="GMM_Regime",p_adjust="holm")
print("\nDUNN POST-HOC (HOLM-ADJUSTED P-VALUES):\n",dunn.round(6).to_string())
dunn.to_csv("Dunn_Holm_Results.csv")

# SIGNIFICANCE LABELS
def sig_label(p):
    return "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else "ns"

# FIGURE 1: HDI DISTRIBUTIONS
fig,ax=plt.subplots(figsize=(11,7))
bp=ax.boxplot(values,positions=np.arange(1,k+1),widths=.55,patch_artist=True,showmeans=True,
              meanprops=dict(marker="D",markerfacecolor="white",markeredgecolor="black",markersize=8),
              medianprops=dict(color="black",linewidth=2),
              whiskerprops=dict(color="black",linewidth=1.3),
              capprops=dict(color="black",linewidth=1.3),
              flierprops=dict(marker="o",markerfacecolor="#FFB703",markeredgecolor="black",markersize=6))
for patch,color in zip(bp["boxes"],COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(.85)
rng=np.random.default_rng(SEED)
for i,(vals,color) in enumerate(zip(values,COLORS),1):
    x=rng.normal(i,.055,len(vals))
    ax.scatter(x,vals,s=30,color=color,edgecolor="black",linewidth=.35,alpha=.65,zorder=3)
    row=profile.loc[groups[i-1]]
    ax.text(i,row["max"]+.025,f"N={int(row['count'])}\nMean={row['mean']:.3f}\nMedian={row['median']:.3f}",
            ha="center",va="bottom",fontsize=9,fontweight="bold",color="black")
ax.set_xticks(range(1,k+1),[f"Regime {i}" for i in groups],fontweight="bold")
ax.set_ylabel("Human Development Index (HDI)",fontweight="bold")
ax.set_title(f"HDI Distribution Across GMM Regimes\nKruskal-Wallis H={H:.3f}, p={p:.2e}, ε²={epsilon_sq:.3f}",
             fontsize=13,fontweight="bold")
ax.set_ylim(max(0,data.HDI.min()-.06),min(1.08,data.HDI.max()+.19))
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig.savefig("08_HDI_by_GMM_Regime.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()
plt.close(fig)

# FIGURE 2: DUNN PAIRWISE COMPARISONS
pairs=[(groups[i],groups[j]) for i in range(len(groups)) for j in range(i+1,len(groups))]
fig,ax=plt.subplots(figsize=(11,6))
y_base=min(1.05,data.HDI.max()+.08)
step=.065
for idx,(a,b) in enumerate(pairs):
    pv=float(dunn.loc[a,b])
    y=y_base+idx*step
    x1,x2=groups.index(a)+1,groups.index(b)+1
    ax.plot([x1,x1,x2,x2],[y-.012,y,y,y-.012],color="black",linewidth=1.4)
    label=f"p={pv:.2e} {sig_label(pv)}"
    ax.text((x1+x2)/2,y+.008,label,ha="center",va="bottom",fontsize=9,fontweight="bold",color="#222222")
ax.set_xlim(.5,k+.5)
ax.set_xticks(range(1,k+1),[f"Regime {i}" for i in groups],fontweight="bold")
ax.set_ylabel("HDI",fontweight="bold")
ax.set_ylim(data.HDI.min()-.04,y_base+len(pairs)*step+.06)
ax.set_title("Dunn Pairwise Comparisons (Holm-Adjusted)",fontweight="bold",fontsize=13)
ax.text(.02,.97,f"Kruskal-Wallis: H={H:.3f}, p={p:.2e}",transform=ax.transAxes,
        ha="left",va="top",fontsize=10,fontweight="bold",
        bbox=dict(boxstyle="round,pad=.4",facecolor="#FFF3B0",edgecolor="black"))
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig.savefig("09_Dunn_Holm_Comparisons.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()
plt.close(fig)

# FIGURE 3: DUNN P-VALUE HEATMAP
annot=dunn.copy().astype(object)
for i in range(len(groups)):
    for j in range(len(groups)):
        pv=float(dunn.iloc[i,j])
        annot.iloc[i,j]="—" if i==j else f"{pv:.2e}\n{sig_label(pv)}"
fig,ax=plt.subplots(figsize=(8,6))
sns.heatmap(dunn.astype(float),annot=annot,fmt="",cmap="YlOrRd_r",vmin=0,vmax=.05,
            linewidths=1,linecolor="white",square=True,
            cbar_kws={"label":"Holm-adjusted p-value (color scale: 0–0.05)"},ax=ax)
ax.set_xticklabels([f"Regime {i}" for i in groups],fontweight="bold")
ax.set_yticklabels([f"Regime {i}" for i in groups],rotation=0,fontweight="bold")
ax.set_title("Pairwise HDI Differences: Dunn–Holm Test",fontweight="bold",fontsize=13)
fig.tight_layout()
fig.savefig("10_Dunn_Holm_Heatmap.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()
plt.close(fig)

print("\nANALYSIS COMPLETED.")
print("Figures: 08_HDI_by_GMM_Regime.png, 09_Dunn_Holm_Comparisons.png, 10_Dunn_Holm_Heatmap.png")
print("Table: Dunn_Holm_Results.csv")


#10. Fuzzy C-Means: belirsiz üyelik analizi
Fuzzy C-Means her ülkeye tek bir kesin sınıf vermek yerine, her rejime ait üyelik derecesi hesaplar. Aşağıdaki uygulama scikit-fuzzy bağımlılığı olmadan çalışır.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score

def fuzzy_cmeans(X,k,m=2,max_iter=1000,tol=1e-6,n_init=30,seed=42):
    X=np.asarray(X,dtype=float)
    if X.ndim!=2 or not np.isfinite(X).all(): raise ValueError("X geçerli ve sonlu iki boyutlu veri olmalı.")
    if k<2 or k>=len(X) or m<=1: raise ValueError("k veya m değeri geçersiz.")
    best=None
    for s in range(n_init):
        rng=np.random.default_rng(seed+s)
        U=rng.dirichlet(np.ones(k),size=len(X))
        for it in range(max_iter):
            Um=U**m
            centers=(Um.T@X)/Um.sum(axis=0)[:,None]
            D=np.maximum(np.linalg.norm(X[:,None,:]-centers[None,:,:],axis=2),1e-12)
            zero=D<1e-11
            Unew=np.zeros_like(U)
            for i in range(len(X)):
                if zero[i].any(): Unew[i,np.where(zero[i])[0]]=1/zero[i].sum()
                else:
                    ratios=(D[i,:,None]/D[i,None,:])**(2/(m-1))
                    Unew[i]=1/ratios.sum(axis=1)
            diff=np.max(np.abs(Unew-U))
            U=Unew
            if diff<tol: break
        Um=U**m
        centers=(Um.T@X)/Um.sum(axis=0)[:,None]
        D=np.linalg.norm(X[:,None,:]-centers[None,:,:],axis=2)
        objective=float(np.sum((U**m)*D**2))
        if best is None or objective<best["objective"]:
            best={"centers":centers.copy(),"U":U.copy(),"iterations":it+1,"objective":objective}
    return best

# FCM ESTIMATION
Z_array=np.asarray(Z,dtype=float)
if len(df)!=len(Z_array): raise ValueError("df ve Z satır sayıları eşleşmiyor.")
fcm=fuzzy_cmeans(Z_array,best_k,m=2,n_init=30,seed=SEED)
fcm_centers,U=fcm["centers"],fcm["U"]
fcm_labels=U.argmax(axis=1)+1
df=df.copy()
df["FCM_Regime"]=fcm_labels
df["FCM_Confidence"]=U.max(axis=1)
fcm_entropy=-np.sum(U*np.log(np.maximum(U,1e-12)),axis=1)/np.log(best_k)
df["FCM_Entropy"]=fcm_entropy

# PROFILES AND COMPARISON
fcm_profile=df.groupby("FCM_Regime")[features+["HDI"]].mean()
fcm_sizes=df["FCM_Regime"].value_counts().sort_index()
ari=adjusted_rand_score(gmm_labels,fcm_labels-1)
nmi=normalized_mutual_info_score(gmm_labels,fcm_labels-1)
print("FCM iterations:",fcm["iterations"])
print("FCM objective:",round(fcm["objective"],6))
print("Mean maximum membership:",round(U.max(axis=1).mean(),4))
print("Mean normalized entropy:",round(fcm_entropy.mean(),4))
print("GMM-FCM ARI:",round(ari,4))
print("GMM-FCM NMI:",round(nmi,4))
print("\nFCM REGIME SIZES:\n",fcm_sizes.to_string())
print("\nFCM REGIME PROFILES:\n",fcm_profile.round(3).to_string())

# VISUALIZATION
sns.set_theme(style="white",font="Arial")
fig,axs=plt.subplots(1,2,figsize=(15,6))
im=axs[0].imshow(U,cmap="viridis",aspect="auto",vmin=0,vmax=1)
axs[0].set_xlabel("Fuzzy regime",fontweight="bold")
axs[0].set_ylabel("Country observation",fontweight="bold")
axs[0].set_xticks(range(best_k),[f"Regime {i}" for i in range(1,best_k+1)],fontweight="bold")
axs[0].set_title("FCM Membership Matrix",fontweight="bold")
fig.colorbar(im,ax=axs[0],label="Membership probability")
bars=axs[1].bar(range(1,best_k+1),U.mean(axis=0),color=COLORS[:best_k],edgecolor="black")
axs[1].set_xlabel("Fuzzy regime",fontweight="bold")
axs[1].set_ylabel("Mean membership",fontweight="bold")
axs[1].set_title("Average Membership by Regime",fontweight="bold")
for bar,v in zip(bars,U.mean(axis=0)):
    axs[1].text(bar.get_x()+bar.get_width()/2,v+.01,f"{v:.3f}",ha="center",fontweight="bold")
axs[1].set_ylim(0,1)
fig.tight_layout()
finish(fig,"09_fcm_memberships.png")


In [ ]:
print(df.columns.tolist())

In [ ]:
# PROFILES AND COMPARISON
fcm_profile=df.groupby("FCM_Regime")[features+["HDI"]].mean()
fcm_sizes=df["FCM_Regime"].value_counts().sort_index()

gmm_labels=df["GMM_Regime"].to_numpy()
fcm_labels=df["FCM_Regime"].to_numpy()

ari=adjusted_rand_score(gmm_labels,fcm_labels)
nmi=normalized_mutual_info_score(gmm_labels,fcm_labels)

print("FCM iterations:",fcm["iterations"])
print("FCM objective:",round(fcm["objective"],6))
print("Mean maximum membership:",round(U.max(axis=1).mean(),4))
print("Mean normalized entropy:",round(fcm_entropy.mean(),4))
print("GMM-FCM ARI:",round(ari,4))
print("GMM-FCM NMI:",round(nmi,4))

print("\nFCM REGIME SIZES:\n",fcm_sizes.to_string())
print("\nFCM REGIME PROFILES:\n",fcm_profile.round(3).to_string())

In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

# 1. DATA VALIDATION
if "df" not in globals(): raise ValueError("Önce ana analiz kodunu çalıştırarak df veri setini oluşturun.")
required=["GMM_Regime","FCM_Regime"]
for col in required:
    if col not in df.columns: raise ValueError(f"df içerisinde {col} sütunu bulunamadı.")
data=df[required].dropna().copy()
if len(data)<3: raise ValueError("Bootstrap analizi için yeterli gözlem bulunmuyor.")
gmm_labels=data["GMM_Regime"].to_numpy()
base_labels=data["FCM_Regime"].to_numpy()
if len(np.unique(gmm_labels))<2 or len(np.unique(base_labels))<2: raise ValueError("Her iki yöntemde de en az iki rejim bulunmalıdır.")

# 2. MEMBERSHIP SENSITIVITY
S=pd.DataFrame({"m":[1.5,2.0,2.5],"Membership":[.8856,.7568,.6545],"Entropy":[.2649,.5585,.7393],"ARI":[.9192,1.0000,.9674]})

# 3. GMM-FCM CROSS-TABULATION
ct=pd.crosstab(pd.Series(gmm_labels,name="GMM_Regime"),pd.Series(base_labels,name="FCM_Regime"))
M=ct.to_numpy()

# 4. REGIME LABEL MATCHING
row_ind,col_ind=linear_sum_assignment(-M)
gmm_values,fcm_values=ct.index.to_numpy(),ct.columns.to_numpy()
mapping={fcm_values[j]:gmm_values[i] for i,j in zip(row_ind,col_ind)}
matched=np.array([mapping.get(label,label) for label in base_labels])
matched_agreement=np.mean(matched==gmm_labels)

# 5. ORIGINAL GMM-FCM AGREEMENT
original_ari=adjusted_rand_score(gmm_labels,base_labels)
original_nmi=normalized_mutual_info_score(gmm_labels,base_labels)

# 6. BOOTSTRAP ANALYSIS
B,SEED=500,42
rng=np.random.default_rng(SEED)
boot_results=[]
n=len(gmm_labels)
for b in range(B):
    indices=rng.choice(n,size=n,replace=True)
    gmm_sample,fcm_sample=gmm_labels[indices],base_labels[indices]
    boot_results.append({"ARI":adjusted_rand_score(gmm_sample,fcm_sample),"NMI":normalized_mutual_info_score(gmm_sample,fcm_sample)})
boot=pd.DataFrame(boot_results)

# 7. BOOTSTRAP CONFIDENCE INTERVALS
ari_mean,ari_std=boot["ARI"].mean(),boot["ARI"].std()
nmi_mean,nmi_std=boot["NMI"].mean(),boot["NMI"].std()
ari_lo,ari_hi=np.percentile(boot["ARI"],[2.5,97.5])
nmi_lo,nmi_hi=np.percentile(boot["NMI"],[2.5,97.5])

# 8. VISUALIZATION SETTINGS
sns.set_theme(style="white",font="Arial")
plt.rcParams.update({"font.family":"Arial","font.size":13,"font.weight":"bold","text.color":"black","axes.labelcolor":"black","axes.titlecolor":"black","axes.edgecolor":"black","axes.linewidth":1.5,"xtick.color":"black","ytick.color":"black","figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white"})
fig,ax=plt.subplots(1,3,figsize=(21,7),gridspec_kw={"width_ratios":[1.15,1.2,1]})

# 9. PANEL A: MEMBERSHIP SENSITIVITY
a=ax[0];x=np.arange(3);w=.24
colors=["#00A651","#FF8C00","#1769FF"]
for j,col in enumerate(["Membership","Entropy","ARI"]):
    bars=a.bar(x+(j-1)*w,S[col],w,color=colors[j],edgecolor="black",linewidth=.8,label=col)
    for bar,v in zip(bars,S[col]): a.text(bar.get_x()+bar.get_width()/2,v+.025,f"{v:.3f}",ha="center",va="bottom",fontsize=11,fontweight="bold",color="black")
a.set_xticks(x,["m = 1.5","m = 2.0","m = 2.5"]);a.set_ylim(0,1.18)
a.set_ylabel("Metric value",fontsize=14,fontweight="bold")
a.set_title("A. Membership sensitivity",fontsize=16,pad=14,fontweight="bold")
a.legend(frameon=False,fontsize=10);a.grid(axis="y",alpha=.22);a.set_axisbelow(True)

# 10. PANEL B: BOOTSTRAP STABILITY
a=ax[1];bins=np.linspace(0,1,26)
a.hist(boot["ARI"],bins=bins,color="#00A651",alpha=.82,edgecolor="black",linewidth=.5,label="ARI")
a.hist(boot["NMI"],bins=bins,color="#1769FF",alpha=.65,edgecolor="black",linewidth=.5,label="NMI")
for col,color in [("ARI","#006B35"),("NMI","#003C99")]:
    mu=boot[col].mean()
    a.axvline(mu,color=color,linestyle="--",linewidth=2.5,label=f"{col} mean = {mu:.3f}")
    a.text(mu,.96,f"{mu:.3f}",transform=a.get_xaxis_transform(),ha="center",va="top",fontsize=11,fontweight="bold",color="black",bbox={"facecolor":"white","edgecolor":color,"alpha":.95})
a.set_xlabel("Stability coefficient",fontsize=14,fontweight="bold")
a.set_ylabel("Bootstrap frequency",fontsize=14,fontweight="bold")
a.set_title("B. Bootstrap stability (B = 500)",fontsize=16,pad=14,fontweight="bold")
a.legend(frameon=False,fontsize=10);a.grid(axis="y",alpha=.22);a.set_axisbelow(True)

# 11. PANEL C: GMM-FCM CORRESPONDENCE
a=ax[2];row_pct=M/M.sum(axis=1,keepdims=True)*100
im=a.imshow(row_pct,cmap="turbo",vmin=0,vmax=100,aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        a.text(j,i,f"{M[i,j]}\n{row_pct[i,j]:.1f}%",ha="center",va="center",fontsize=13,fontweight="bold",color="black",bbox={"facecolor":"white","alpha":.78,"edgecolor":"none","boxstyle":"round,pad=.22"})
a.set_xticks(range(len(ct.columns)),[f"FCM {v}" for v in ct.columns],fontsize=12,fontweight="bold")
a.set_yticks(range(len(ct.index)),[f"GMM {v}" for v in ct.index],fontsize=12,fontweight="bold")
a.set_xlabel("FCM regimes",fontsize=14,fontweight="bold")
a.set_ylabel("GMM regimes",fontsize=14,fontweight="bold")
a.set_title("C. GMM-FCM correspondence",fontsize=16,pad=14,fontweight="bold")
cb=fig.colorbar(im,ax=a,fraction=.046,pad=.04)
cb.set_label("Within-GMM regime (%)",fontsize=12,fontweight="bold")
cb.ax.tick_params(labelsize=11,colors="black")

# 12. FRAME AND TICKS
for a in ax:
    a.tick_params(axis="both",labelsize=12,width=1.4,length=5)
    for sp in a.spines.values(): sp.set_linewidth(1.5);sp.set_color("black")

# 13. NUMERICAL SUMMARY
summary=(f"Bootstrap ARI: {ari_mean:.4f} ± {ari_std:.4f} | 95% CI [{ari_lo:.4f}, {ari_hi:.4f}]     "
         f"Bootstrap NMI: {nmi_mean:.4f} ± {nmi_std:.4f} | 95% CI [{nmi_lo:.4f}, {nmi_hi:.4f}]\n"
         f"GMM-FCM ARI = {original_ari:.4f} | NMI = {original_nmi:.4f} | Matched agreement = {matched_agreement:.4f}")
fig.suptitle("FCM Robustness, Bootstrap Stability and GMM Comparison",fontsize=19,fontweight="bold",color="black",y=1.02)
fig.text(.5,.015,summary,ha="center",va="bottom",fontsize=11,fontweight="bold",color="black")
fig.tight_layout(rect=[0,.10,1,.96])

# 14. SAVE FIGURES
fig.savefig("FCM_Robustness_Visualization.png",dpi=600,bbox_inches="tight")
fig.savefig("FCM_Robustness_Visualization.pdf",bbox_inches="tight")
plt.show()

# 15. PRINT RESULTS
print("\nSENSITIVITY RESULTS\n",S.round(4).to_string(index=False))
print("\nBOOTSTRAP SUMMARY\n",boot[["ARI","NMI"]].describe().round(4))
print(f"\nARI 95% CI: [{ari_lo:.4f}, {ari_hi:.4f}]")
print(f"NMI 95% CI: [{nmi_lo:.4f}, {nmi_hi:.4f}]")
print("\nGMM-FCM CROSS-TABULATION\n",ct)
print("\nGMM-FCM ARI:",round(original_ari,4))
print("GMM-FCM NMI:",round(original_nmi,4))
print("MATCHED AGREEMENT:",round(matched_agreement,4))
print("\nOUTPUT FILES:\nFCM_Robustness_Visualization.png\nFCM_Robustness_Visualization.pdf")


In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score

data=df[["GMM_Regime","FCM_Regime"]].dropna()
ct=pd.crosstab(data["GMM_Regime"],data["FCM_Regime"])
row_pct=ct.div(ct.sum(axis=1),axis=0)*100

fig,ax=plt.subplots(figsize=(8,6))
sns.heatmap(row_pct,annot=np.array([[f"{ct.iloc[i,j]}\n({row_pct.iloc[i,j]:.1f}%)" for j in range(ct.shape[1])] for i in range(ct.shape[0])]),fmt="",cmap="YlGnBu",vmin=0,vmax=100,linewidths=1,linecolor="black",cbar_kws={"label":"Within-GMM regime (%)"},annot_kws={"fontsize":12,"fontweight":"bold","color":"black"},ax=ax)
ax.set_xlabel("FCM regimes",fontweight="bold",fontsize=13)
ax.set_ylabel("GMM regimes",fontweight="bold",fontsize=13)
ax.set_title("GMM–FCM Regime Correspondence",fontweight="bold",fontsize=15,pad=12)
ax.set_xticklabels([f"FCM {x}" for x in ct.columns],fontweight="bold")
ax.set_yticklabels([f"GMM {x}" for x in ct.index],fontweight="bold",rotation=0)
plt.tight_layout()
fig.savefig("Figure_10_GMM_FCM_Correspondence.png",dpi=600,bbox_inches="tight")
fig.savefig("Figure_10_GMM_FCM_Correspondence.pdf",bbox_inches="tight")
plt.show()
print("ARI:",round(adjusted_rand_score(data["GMM_Regime"],data["FCM_Regime"]),4))
print("NMI:",round(normalized_mutual_info_score(data["GMM_Regime"],data["FCM_Regime"]),4))
print("Cross-tabulation:\n",ct)

11. Archetypal Analysis: karakteristik ülke profilleri
Bu aşamada her ülke, belirlenen arketiplerin ağırlıklı bileşimi olarak temsil edilir. Arketipler gözlenen ülkelerin konveks kombinasyonlarıdır. Optimizasyon başlangıç değerlerine duyarlı olabileceğinden bu sonuçları tamamlayıcı ve keşfedici analiz olarak yorumlamalısın.

In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import softmax
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score

SEED=42;N_INIT=100;MAXITER=10000;K_AA=3
X=np.asarray(Z,dtype=float);n,p=X.shape
if not np.isfinite(X).all() or len(df)!=n: raise ValueError("Z ve df satırları eşleşmeli ve sonlu değerlerden oluşmalı.")
if len(gmm_labels)!=n or len(base_labels)!=n: raise ValueError("GMM ve FCM etiketlerinin uzunluğu Z ile eşleşmeli.")
pc_names=[f"PC{i+1}" for i in range(p)]

def archetypal_analysis(X,k,seed=42,maxiter=10000,n_init=100):
    n,p=X.shape;results=[];best=None
    for s in range(n_init):
        rng=np.random.default_rng(seed+s)
        A0=rng.dirichlet(np.ones(k),size=n);B0=rng.dirichlet(np.ones(n),size=k)
        def objective(theta):
            A=softmax(theta[:n*k].reshape(n,k),axis=1)
            B=softmax(theta[n*k:].reshape(k,n),axis=1)
            R=A@(B@X)-X;loss=np.sum(R**2)
            gA=2*R@(B@X).T;gB=2*(A.T@R)@X.T
            gLA=A*(gA-np.sum(gA*A,axis=1,keepdims=True))
            gLB=B*(gB-np.sum(gB*B,axis=1,keepdims=True))
            return loss,np.r_[gLA.ravel(),gLB.ravel()]
        theta0=np.r_[np.log(A0+1e-12).ravel(),np.log(B0+1e-12).ravel()]
        res=minimize(objective,theta0,jac=True,method="L-BFGS-B",options={"maxiter":maxiter,"ftol":1e-9,"maxls":50})
        results.append({"Start":s+1,"SSE":res.fun,"RMSE":np.sqrt(res.fun/(n*p)),"Success":res.success,"Iterations":res.nit,"Message":res.message})
        if np.isfinite(res.fun) and (best is None or res.fun<best["SSE"]):
            A=softmax(res.x[:n*k].reshape(n,k),axis=1)
            B=softmax(res.x[n*k:].reshape(k,n),axis=1)
            best={"A":A,"B":B,"archetypes":B@X,"SSE":float(res.fun),"result":res}
    return best,pd.DataFrame(results)

# FIT K=3
aa,runs=archetypal_analysis(X,K_AA,SEED,MAXITER,N_INIT)
if aa is None: raise RuntimeError("Geçerli AA çözümü bulunamadı.")
A=aa["A"];B=aa["B"];archetypes=aa["archetypes"]
aa_labels=A.argmax(axis=1)+1
rmse=np.sqrt(aa["SSE"]/(n*p))
r2=1-aa["SSE"]/np.sum((X-X.mean(axis=0))**2)
entropy=-np.sum(A*np.log(np.maximum(A,1e-12)),axis=1)/np.log(K_AA)

# OPTIMIZATION DIAGNOSTICS
print("\nARCHETYPAL ANALYSIS: K=3")
print("Successful starts:",runs.Success.sum(),"/",N_INIT)
print("Best SSE:",round(aa["SSE"],6))
print("Reconstruction RMSE:",round(rmse,6))
print("Reconstruction R2:",round(r2,6))
print("Best optimization success:",aa["result"].success)
print("Best iterations:",aa["result"].nit)
print("Best message:",aa["result"].message)
print("Mean maximum membership:",round(A.max(axis=1).mean(),4))
print("Mean normalized entropy:",round(entropy.mean(),4))
print("\nBEST 10 STARTS:\n",runs.sort_values("SSE").head(10).round(5).to_string(index=False))
print("\nARCHETYPE COORDINATES:\n",pd.DataFrame(archetypes,index=[f"Archetype {i}" for i in range(1,K_AA+1)],columns=pc_names).round(4))
print("\nARCHETYPE MEMBERSHIP SIZES:\n",pd.Series(aa_labels).value_counts().sort_index().to_string())

# COUNTRY-LEVEL RESULTS
membership=pd.DataFrame(A,columns=[f"AA_Archetype_{i}" for i in range(1,K_AA+1)])
membership.insert(0,"Country",df["Country"].astype(str).to_numpy())
membership["AA_Regime"]=aa_labels
membership["Max_membership"]=A.max(axis=1)
membership["Normalized_entropy"]=entropy
membership.to_csv("AA_K3_Country_Results.csv",index=False)
pd.DataFrame(archetypes,index=[f"Archetype {i}" for i in range(1,K_AA+1)],columns=pc_names).to_csv("AA_K3_Archetype_Coordinates.csv")
runs.to_csv("AA_K3_Optimization_Runs.csv",index=False)

# GMM-FCM-AA COMPARISON
print("\nGMM-FCM ARI:",round(adjusted_rand_score(gmm_labels,base_labels),4))
print("GMM-AA ARI:",round(adjusted_rand_score(gmm_labels,aa_labels),4))
print("FCM-AA ARI:",round(adjusted_rand_score(base_labels,aa_labels),4))
print("GMM-AA NMI:",round(normalized_mutual_info_score(gmm_labels,aa_labels),4))
print("FCM-AA NMI:",round(normalized_mutual_info_score(base_labels,aa_labels),4))

# PUBLICATION-QUALITY FIGURES
plt.rcParams.update({"font.family":"Arial","font.size":13,"font.weight":"bold","text.color":"black","axes.labelcolor":"black","axes.titlecolor":"black","xtick.color":"black","ytick.color":"black","axes.edgecolor":"black","axes.linewidth":1.4,"figure.facecolor":"white","axes.facecolor":"white"})

# FIGURE 1: OPTIMIZATION DIAGNOSTICS
fig,ax=plt.subplots(1,2,figsize=(13,5))
bars=ax[0].bar(["Successful","Unsuccessful"],[runs.Success.sum(),N_INIT-runs.Success.sum()],color=["#00A86B","#D62755"],edgecolor="black")
for b,v in zip(bars,[runs.Success.sum(),N_INIT-runs.Success.sum()]):
    ax[0].text(b.get_x()+b.get_width()/2,v+1,str(v),ha="center",fontweight="bold")
ax[0].set_ylabel("Number of initializations");ax[0].set_title("A. Optimization convergence")
ax[0].set_ylim(0,N_INIT*1.15);ax[0].grid(axis="y",alpha=.2)
ax[1].hist(runs.SSE,bins=20,color="#1677FF",edgecolor="black")
ax[1].axvline(aa["SSE"],color="#D62755",ls="--",lw=2,label=f"Best SSE = {aa['SSE']:.2f}")
ax[1].set_xlabel("Reconstruction SSE");ax[1].set_ylabel("Frequency")
ax[1].set_title("B. Multi-start objective distribution");ax[1].legend(frameon=False)
fig.suptitle("Archetypal Analysis (K=3): Optimization Diagnostics",fontsize=17,fontweight="bold")
fig.tight_layout()
fig.savefig("AA_K3_Optimization.png",dpi=600,bbox_inches="tight")
fig.savefig("AA_K3_Optimization.pdf",bbox_inches="tight")
plt.show()

# FIGURE 2: MEMBERSHIP HEATMAP WITH LABELS
fig,ax=plt.subplots(figsize=(10,6))
order=np.argsort(A.max(axis=1))[::-1]
im=ax.imshow(A[order],aspect="auto",cmap="turbo",vmin=0,vmax=1,interpolation="nearest")
ax.set_xlabel("Archetypes");ax.set_ylabel("Country observations (sorted by maximum membership)")
ax.set_xticks(range(K_AA),[f"Archetype {i}" for i in range(1,K_AA+1)])
ax.set_title("Country Membership in Archetypes (K=3)",fontsize=16)
for i in range(len(order)):
    for j in range(K_AA):
        ax.text(j,i,f"{A[order[i],j]:.2f}",ha="center",va="center",color="black",fontsize=7,fontweight="bold")
cb=fig.colorbar(im,ax=ax);cb.set_label("Membership weight")
fig.tight_layout()
fig.savefig("AA_K3_Membership.png",dpi=600,bbox_inches="tight")
fig.savefig("AA_K3_Membership.pdf",bbox_inches="tight")
plt.show()

# FIGURE 3: ARCHETYPAL PROFILES WITH ROTATED LABELS
fig,ax=plt.subplots(figsize=(10,6))
xx=np.arange(K_AA);width=.8/p
for j in range(p):
    bars=ax.bar(xx+(j-(p-1)/2)*width,archetypes[:,j],width,label=pc_names[j],color=plt.cm.turbo(j/max(p-1,1)),edgecolor="black",linewidth=.6)
    for b,v in zip(bars,archetypes[:,j]):
        ax.annotate(f"{v:.3f}",(b.get_x()+b.get_width()/2,v),xytext=(0,3 if v>=0 else -3),textcoords="offset points",ha="center",va="bottom" if v>=0 else "top",rotation=90,color="black",fontsize=10,fontweight="bold")
ax.set_xticks(xx,[f"Archetype {i}" for i in range(1,K_AA+1)])
ax.set_ylabel("PCA score");ax.set_title("Archetypal Profiles in PCA Space",fontsize=16)
ax.axhline(0,color="black",lw=.8);ax.legend(ncol=p,frameon=False)
fig.tight_layout()
fig.savefig("AA_K3_Profiles.png",dpi=600,bbox_inches="tight")
fig.savefig("AA_K3_Profiles.pdf",bbox_inches="tight")
plt.show()


 # 4. Şimdi yapmamız gereken kritik analiz
Ben makalenin nihai sonuçlarını yazmadan önce iki kontrolü zorunlu görüyorum:

Birincisi: Arketipleri orijinal sekiz değişken üzerinden yorumlamak.
 
PC1–PC4 koordinatları tek başına ekonomik yorum için yeterli değil. PCA ters dönüşümüyle arketiplerin HDI, LE, EYS, MYS, GNI, CHE, CO2 ve EA değerlerini elde etmeliyiz.

PCA nesnen pca, değişken sıralaman features ise şu kodu kullanabilirsin:

In [ ]:
# ARCHETYPE COMPOSITION AND COUNTRY DIAGNOSTICS

import numpy as np
import pandas as pd

A=aa["A"]
B=aa["B"]
archetypes=aa["archetypes"]
features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]
country_names=df["Country"].astype(str).to_numpy()
K=A.shape[1]
p=archetypes.shape[1]

if len(country_names)!=A.shape[0] or B.shape!=(K,len(country_names)):
    raise ValueError("A, B ve df boyutları uyumsuz.")
if len(features)!=pca.n_features_in_:
    raise ValueError("Features ve PCA giriş değişkenleri uyuşmuyor.")
if p>pca.n_components_:
    raise ValueError("AA bileşen sayısı PCA bileşen sayısından büyük.")

aa_labels=A.argmax(axis=1)+1

for j in range(K):
    print(f"\n{'='*55}\nARCHETYPE {j+1}")
    print("Assigned countries:",np.sum(aa_labels==j+1))

    idx=np.argsort(A[:,j])[::-1][:10]
    print("\nTop 10 membership countries:")
    print(pd.DataFrame({"Country":country_names[idx],"Membership":A[idx,j]}).to_string(index=False))

    idxb=np.argsort(B[j])[::-1][:10]
    print("\nTop 10 archetype-defining countries:")
    print(pd.DataFrame({"Country":country_names[idxb],"B_weight":B[j,idxb]}).to_string(index=False))

print("\nCOUNTRIES ASSIGNED TO ARCHETYPE 1:")
idx=np.where(aa_labels==1)[0]
print(pd.DataFrame({"Country":country_names[idx],"Membership":A[idx].max(axis=1)}).to_string(index=False))

print("\nMEMBERSHIP DISTRIBUTION:")
print(pd.DataFrame(A,columns=[f"Archetype_{i}" for i in range(1,K+1)]).describe().round(4))

# APPROXIMATE ARCHETYPE PROFILES

scores=np.zeros((K,pca.n_components_))
scores[:,:p]=archetypes
archetype_profiles=pca.inverse_transform(scores)

profiles=pd.DataFrame(archetype_profiles,columns=features,index=[f"Archetype {i}" for i in range(1,K+1)])

print("\nAPPROXIMATE ARCHETYPE PROFILES:")
print(profiles.round(3).to_string())

In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.optimize import minimize,linear_sum_assignment
from scipy.special import softmax

SEED=42;K=3;N_BOOT=200;N_INIT_BOOT=3;MAXITER_BOOT=800
features=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
Z0=np.asarray(Z,dtype=float).copy();n,p=Z0.shape
Y=df[features].astype(float).to_numpy()
if p!=4 or Y.shape!=(n,8): raise ValueError("Z veya orijinal veri boyutları hatalı.")
if not np.isfinite(Z0).all() or not np.isfinite(Y).all(): raise ValueError("Eksik veya sonsuz değer var.")
if len(df)!=n: raise ValueError("df ve Z satırları eşleşmiyor.")
if "Country" not in df.columns: raise ValueError("Country sütunu bulunamadı.")
if "archetypes" not in globals() or np.asarray(archetypes).shape!=(K,p): raise ValueError("K=3 archetypes bulunamadı.")
reference=np.asarray(archetypes,dtype=float).copy()

# LINEAR PROJECTION
D=np.column_stack([np.ones(n),Z0])
coef=np.linalg.lstsq(D,Y,rcond=None)[0]
Y_hat=D@coef
r2=1-np.sum((Y-Y_hat)**2,axis=0)/np.sum((Y-Y.mean(axis=0))**2,axis=0)
rmse=np.sqrt(np.mean((Y-Y_hat)**2,axis=0))
mapping_quality=pd.DataFrame({"Variable":features,"R2":r2,"RMSE":rmse})
print("\nLINEAR MAPPING VALIDATION\n",mapping_quality.round(4).to_string(index=False))
mapping_quality.to_csv("AA_K3_Linear_Mapping_Validation.csv",index=False)

def original_scale(ar):
    ar=np.asarray(ar,dtype=float).reshape(-1,p)
    return np.column_stack([np.ones(len(ar)),ar])@coef

baseline_values=original_scale(reference)
baseline_profiles=pd.DataFrame(baseline_values,columns=features,index=[f"Archetype {i}" for i in range(1,K+1)])
print("\nBASELINE ARCHETYPE PROFILES\n",baseline_profiles.round(3).to_string())
baseline_profiles.to_csv("AA_K3_Original_Scale_Profiles.csv")

# ARCHETYPAL ANALYSIS
def fit_aa(data,k,seed,maxiter,n_init):
    nn,pp=data.shape;best=None
    for s in range(n_init):
        rng=np.random.default_rng(seed+s)
        A0=rng.dirichlet(np.ones(k),size=nn)
        B0=rng.dirichlet(np.ones(nn),size=k)
        def objective(t):
            A=softmax(t[:nn*k].reshape(nn,k),axis=1)
            B=softmax(t[nn*k:].reshape(k,nn),axis=1)
            BD=B@data;R=A@BD-data
            gA=2*R@BD.T
            gB=2*(A.T@R)@data.T
            gLA=A*(gA-np.sum(gA*A,axis=1,keepdims=True))
            gLB=B*(gB-np.sum(gB*B,axis=1,keepdims=True))
            return np.sum(R**2),np.r_[gLA.ravel(),gLB.ravel()]
        t0=np.r_[np.log(A0+1e-12).ravel(),np.log(B0+1e-12).ravel()]
        res=minimize(objective,t0,jac=True,method="L-BFGS-B",options={"maxiter":maxiter,"ftol":1e-7,"maxls":20})
        if np.isfinite(res.fun) and (best is None or res.fun<best["SSE"]):
            B=softmax(res.x[nn*k:].reshape(k,nn),axis=1)
            best={"archetypes":B@data,"SSE":float(res.fun),"Iterations":res.nit}
    return best

# COUNTRY-LEVEL BOOTSTRAP
rng=np.random.default_rng(SEED)
boot_pc=[];boot_original=[];boot_sse=[];failed=0
for b in range(N_BOOT):
    ix=rng.integers(0,n,n)
    fitted=fit_aa(Z0[ix],K,SEED+1000+b*N_INIT_BOOT,MAXITER_BOOT,N_INIT_BOOT)
    if fitted is None:
        failed+=1
        continue
    candidate=fitted["archetypes"]
    cost=np.linalg.norm(reference[:,None,:]-candidate[None,:,:],axis=2)
    rows,cols=linear_sum_assignment(cost)
    aligned=candidate[cols[np.argsort(rows)]]
    boot_pc.append(aligned)
    boot_original.append(original_scale(aligned))
    boot_sse.append(fitted["SSE"])

if len(boot_pc)<2: raise RuntimeError("Yeterli başarılı bootstrap tekrarı elde edilemedi.")
boot_pc=np.asarray(boot_pc);boot_original=np.asarray(boot_original)
print(f"\nBOOTSTRAP SUCCESS: {len(boot_pc)}/{N_BOOT} ({100*len(boot_pc)/N_BOOT:.1f}%)")
print("BOOTSTRAP FAILED:",failed)

# PCA COORDINATE STABILITY
pc_rows=[]
for j in range(K):
    for q in range(p):
        v=boot_pc[:,j,q]
        pc_rows.append({"Archetype":j+1,"Component":f"PC{q+1}","Mean":v.mean(),"SD":v.std(ddof=1),"CI_2.5":np.percentile(v,2.5),"CI_97.5":np.percentile(v,97.5)})
pc_stability=pd.DataFrame(pc_rows)
print("\nPCA COORDINATE STABILITY\n",pc_stability.round(4).to_string(index=False))
pc_stability.to_csv("AA_K3_Bootstrap_PC_Stability.csv",index=False)

# ORIGINAL VARIABLE PROFILE STABILITY
profile_rows=[]
for j in range(K):
    for q,f in enumerate(features):
        v=boot_original[:,j,q]
        profile_rows.append({"Archetype":j+1,"Variable":f,"Baseline":baseline_profiles.iloc[j,q],"Mean":v.mean(),"SD":v.std(ddof=1),"CI_2.5":np.percentile(v,2.5),"CI_97.5":np.percentile(v,97.5)})
profile_stability=pd.DataFrame(profile_rows)
print("\nORIGINAL VARIABLE PROFILE STABILITY\n",profile_stability.round(3).to_string(index=False))
profile_stability.to_csv("AA_K3_Bootstrap_Original_Profile_CI.csv",index=False)

# ARCHETYPE DISTANCE STABILITY
distances=np.linalg.norm(boot_pc-reference[None,:,:],axis=2)
distance_summary=pd.DataFrame({"Archetype":np.arange(1,K+1),"Mean_distance":distances.mean(axis=0),"SD_distance":distances.std(axis=0,ddof=1),"CI_2.5":np.percentile(distances,2.5,axis=0),"CI_97.5":np.percentile(distances,97.5,axis=0)})
print("\nARCHETYPE DISTANCE STABILITY\n",distance_summary.round(4).to_string(index=False))
distance_summary.to_csv("AA_K3_Bootstrap_Distance.csv",index=False)
print("\nBOOTSTRAP SSE SUMMARY\n",pd.Series(boot_sse,name="SSE").describe().round(4).to_string())

# ORIGINAL-SCALE PROFILE FIGURE
plt.rcParams.update({"font.family":"Arial","font.size":11,"font.weight":"bold","text.color":"black","axes.labelcolor":"black","axes.titlecolor":"black","xtick.color":"black","ytick.color":"black","axes.edgecolor":"black","axes.linewidth":1.2,"figure.facecolor":"white","axes.facecolor":"white"})
fig,ax=plt.subplots(1,K,figsize=(19,7))
colors=["#00A86B","#1677FF","#FF8C00","#D62755","#9467BD","#17BECF","#E377C2","#8C564B"]
for j in range(K):
    vals=baseline_profiles.iloc[j].to_numpy()
    ax[j].bar(features,vals,color=colors,edgecolor="black",linewidth=.7)
    for i,v in enumerate(vals):
        ax[j].text(i,v,f"{v:.2f}",ha="center",va="bottom" if v>=0 else "top",fontsize=8,fontweight="bold",rotation=90)
    ax[j].set_title(f"Archetype {j+1}",fontsize=15,fontweight="bold")
    ax[j].tick_params(axis="x",rotation=45)
    ax[j].grid(axis="y",alpha=.2);ax[j].set_axisbelow(True)
fig.suptitle("Archetypal Profiles: Linear Projection to Original Variables",fontsize=16,fontweight="bold")
fig.tight_layout()
fig.savefig("AA_K3_Original_Profiles.png",dpi=600,bbox_inches="tight")
fig.savefig("AA_K3_Original_Profiles.pdf",bbox_inches="tight")
plt.show()

# BOOTSTRAP PROFILE INTERVALS
fig,ax=plt.subplots(1,K,figsize=(19,7))
for j in range(K):
    sub=profile_stability[profile_stability["Archetype"]==j+1]
    y=np.arange(len(features))
    lower=sub["Mean"].to_numpy()-sub["CI_2.5"].to_numpy()
    upper=sub["CI_97.5"].to_numpy()-sub["Mean"].to_numpy()
    ax[j].errorbar(sub["Mean"].to_numpy(),y,xerr=np.vstack([lower,upper]),fmt="o",capsize=5,color="black",ecolor="#1677FF",markersize=7,linewidth=1.8)
    ax[j].set_yticks(y,features)
    ax[j].invert_yaxis()
    ax[j].set_title(f"Archetype {j+1}",fontsize=15,fontweight="bold")
    ax[j].set_xlabel("Original variable scale",fontweight="bold")
    ax[j].grid(axis="x",alpha=.2);ax[j].set_axisbelow(True)
fig.suptitle("Bootstrap 95% Intervals of Archetypal Profiles",fontsize=16,fontweight="bold")
fig.tight_layout()
fig.savefig("AA_K3_Bootstrap_Intervals.png",dpi=600,bbox_inches="tight")
fig.savefig("AA_K3_Bootstrap_Intervals.pdf",bbox_inches="tight")
plt.show()

print("\nALL ANALYSES COMPLETED.")


In [ ]:

import numpy as np,pandas as pd,networkx as nx,matplotlib.pyplot as plt
from matplotlib.lines import Line2D

threshold=.50
COLOR_MAP={"HDI":"#34495E","LE":"#2878B5","EYS":"#36A269","MYS":"#70AD47","GNI":"#E69F00","CHE":"#9467BD","CO2":"#D9534F","EA":"#17A2A4"}

def network_analysis(data,variables,suffix,title):
    corr=data[variables].apply(pd.to_numeric,errors="coerce").corr(method="pearson")
    G=nx.Graph()
    G.add_nodes_from(variables)
    for i,a in enumerate(variables):
        for b in variables[i+1:]:
            r=corr.loc[a,b]
            if pd.notna(r) and abs(r)>=threshold:
                G.add_edge(a,b,weight=abs(r),correlation=r,distance=1/max(abs(r),1e-6))
    centrality=pd.DataFrame({"Degree centrality":nx.degree_centrality(G),"Betweenness centrality":nx.betweenness_centrality(G,weight="distance"),"Weighted degree":dict(G.degree(weight="weight"))}).sort_values("Weighted degree",ascending=False)
    edges=pd.DataFrame([{"Indicator 1":a,"Indicator 2":b,"Correlation":d["correlation"],"Absolute weight":d["weight"]} for a,b,d in G.edges(data=True)]).sort_values("Absolute weight",ascending=False)
    corr.to_csv(f"{suffix}_correlation_matrix.csv",encoding="utf-8-sig")
    centrality.to_csv(f"{suffix}_centrality.csv",encoding="utf-8-sig")
    edges.to_csv(f"{suffix}_edges.csv",index=False,encoding="utf-8-sig")
    print(f"\n{title}")
    print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
    print("\nEDGES:\n",edges.round(3).to_string(index=False))
    print("\nCENTRALITY:\n",centrality.round(3).to_string())
    pos=nx.kamada_kawai_layout(G,weight="distance")
    fig,ax=plt.subplots(figsize=(12,9),facecolor="white")
    ax.set_facecolor("white")
    weighted=dict(G.degree(weight="weight"))
    max_w=max(weighted.values()) if weighted and max(weighted.values())>0 else 1
    node_sizes=[1800+1500*weighted[n]/max_w for n in G.nodes()]
    node_colors=[COLOR_MAP.get(n,"#808080") for n in G.nodes()]
    nx.draw_networkx_nodes(G,pos,node_size=node_sizes,node_color=node_colors,edgecolors="black",linewidths=2,ax=ax)
    nx.draw_networkx_labels(G,pos,font_size=13,font_weight="bold",font_color="white",ax=ax)
    for sign,color in [(1,"#25855A"),(-1,"#C0392B")]:
        selected=[(a,b) for a,b,d in G.edges(data=True) if np.sign(d["correlation"])==sign]
        if selected:
            nx.draw_networkx_edges(G,pos,edgelist=selected,width=[1.2+5*G[a][b]["weight"] for a,b in selected],edge_color=color,alpha=.78,ax=ax)
    edge_labels={(a,b):f"{d['correlation']:+.2f}" for a,b,d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G,pos,edge_labels=edge_labels,font_size=10,font_weight="bold",font_color="#202020",rotate=False,bbox={"facecolor":"white","edgecolor":"none","alpha":.90,"pad":2},ax=ax)
    legend=[Line2D([0],[0],color="#25855A",lw=3,label="Positive correlation"),Line2D([0],[0],color="#C0392B",lw=3,label="Negative correlation")]
    ax.legend(handles=legend,loc="lower center",ncol=2,frameon=False,fontsize=11)
    ax.set_title(title,fontsize=17,fontweight="bold",color="black",pad=24)
    ax.text(.5,1.01,f"Pearson correlation network | |r| ≥ {threshold:.2f}",transform=ax.transAxes,ha="center",fontsize=11,color="#555555")
    ax.axis("off")
    fig.tight_layout()
    finish(fig,f"{suffix}_network.png")
    return G,corr,centrality,edges

features_no_hdi=[v for v in features if v!="HDI"]
G_main,corr_main,centrality_main,edges_main=network_analysis(df,features_no_hdi,"13_main","Indicator Correlation Network (HDI Excluded)")
G_sensitivity,corr_sensitivity,centrality_sensitivity,edges_sensitivity=network_analysis(df,features,"14_sensitivity","Indicator Correlation Network (HDI Included)")


# 13. Bootstrap: GMM rejimlerinin kararlılığı
Bu aşama 1.000 bootstrap örneklemiyle GMM'yi tekrar kurar ve her modelin bütün ülkeler üzerindeki tahminlerini başlangıç modelinin sınıflandırmasıyla karşılaştırır. ARI etiket sıralamasından etkilenmez. Bootstrap tekrarlarında modelin yakınsamaması veya çok küçük kümeler oluşması da ayrıca izlenmelidir

In [ ]:

N_BOOT=1000
base_labels=gmm.predict(Z)
ari_values=[]
failed=0
rng=np.random.default_rng(SEED)

for b in range(N_BOOT):
    idx=rng.integers(0,len(Z),size=len(Z))
    try:
        gm=GaussianMixture(n_components=best_k,covariance_type="full",n_init=3,reg_covar=1e-5,random_state=SEED+b).fit(Z[idx])
        lab=gm.predict(Z)
        if len(np.unique(lab))>1:
            ari_values.append(adjusted_rand_score(base_labels,lab))
        else:
            failed+=1
    except Exception:
        failed+=1

ari_values=np.asarray(ari_values)

if len(ari_values)==0:
    raise RuntimeError("No successful bootstrap comparisons were obtained.")

print(f"Bootstrap repetitions: {N_BOOT}")
print(f"Successful comparisons: {len(ari_values)}")
print(f"Failed/single-cluster fits: {failed}")
print(f"Mean ARI: {ari_values.mean():.4f}")
print(f"Standard deviation: {ari_values.std(ddof=1):.4f}")
print(f"Median ARI: {np.median(ari_values):.4f}")
print(f"95% interval: [{np.quantile(ari_values,.025):.4f}, {np.quantile(ari_values,.975):.4f}]")

pd.DataFrame({"Bootstrap_ARI":ari_values}).to_csv("13_GMM_Bootstrap_ARI.csv",index=False,encoding="utf-8-sig")

fig,ax=plt.subplots(figsize=(9,5),facecolor="white")
ax.hist(ari_values,bins=25,color="#4472C4",edgecolor="black",linewidth=1,alpha=.9)
ax.axvline(ari_values.mean(),color="black",linestyle="--",linewidth=2,label=f"Mean ARI = {ari_values.mean():.3f}")
ax.axvline(np.median(ari_values),color="#C00000",linestyle=":",linewidth=2,label=f"Median ARI = {np.median(ari_values):.3f}")
ax.set_xlabel("Adjusted Rand Index",fontsize=12,fontweight="bold")
ax.set_ylabel("Bootstrap Frequency",fontsize=12,fontweight="bold")
ax.set_title(f"GMM Bootstrap Stability ({N_BOOT} Replications)",fontsize=14,fontweight="bold",color="black")
ax.legend(frameon=False,fontsize=10)
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
finish(fig,"13_bootstrap_ari.png")


In [ ]:
N_BOOT=100
base_labels=gmm.predict(Z)
ari_values=[]
failed=0
rng=np.random.default_rng(SEED)
for b in range(N_BOOT):
    idx=rng.integers(0,len(Z),size=len(Z))
    try:
        gm=GaussianMixture(n_components=3,covariance_type="full",n_init=20,reg_covar=1e-5,random_state=SEED+b,max_iter=500).fit(Z[idx])
        lab=gm.predict(Z)
        if len(np.unique(lab))>1: ari_values.append(adjusted_rand_score(base_labels,lab))
        else: failed+=1
    except Exception: failed+=1
ari_values=np.asarray(ari_values)
if len(ari_values)==0: raise RuntimeError("No successful bootstrap comparisons were obtained.")
print(f"Successful: {len(ari_values)} | Failed: {failed}")
print(f"Mean ARI: {ari_values.mean():.4f} | SD: {ari_values.std(ddof=1):.4f}")
print(f"Median ARI: {np.median(ari_values):.4f}")
print(f"Empirical 95% interval: [{np.quantile(ari_values,.025):.4f}, {np.quantile(ari_values,.975):.4f}]")
pd.DataFrame({"Bootstrap_ARI":ari_values}).to_csv("07_GMM_Bootstrap_ARI_1000.csv",index=False,encoding="utf-8-sig")
fig,ax=plt.subplots(figsize=(9,5),facecolor="white")
ax.hist(ari_values,bins=25,edgecolor="black",linewidth=1,alpha=.9)
ax.axvline(ari_values.mean(),color="black",linestyle="--",linewidth=2,label=f"Mean ARI = {ari_values.mean():.3f}")
ax.axvline(np.median(ari_values),color="#C00000",linestyle=":",linewidth=2,label=f"Median ARI = {np.median(ari_values):.3f}")
ax.set_xlabel("Adjusted Rand Index",fontsize=12,fontweight="bold")
ax.set_ylabel("Bootstrap Frequency",fontsize=12,fontweight="bold")
ax.set_title("GMM Bootstrap Stability (1,000 Replications)",fontsize=14,fontweight="bold")
ax.legend(frameon=False,fontsize=10)
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
finish(fig,"07_GMM_Bootstrap_ARI_1000.png")

# 14. Ham veriler ve log dönüşümlü verilerin karşılaştırılması
Bu sağlamlık kontrolünde GNI ve CO₂'nin ham değerleri standardize edilerek alternatif bir GMM modeli kurulur. İki çözümün ARI ve NMI değerleri karşılaştırılır. Burada aynı küme sayısı kullanılıyor; böylece dönüşümün sınıflandırma üzerindeki etkisi daha doğrudan inceleniyor.

In [ ]:
X_raw=StandardScaler().fit_transform(df[features])
Z_raw=PCA(n_components=n_pc).fit_transform(X_raw)
gmm_raw=GaussianMixture(n_components=best_k,covariance_type="full",n_init=20,reg_covar=1e-5,random_state=SEED).fit(Z_raw)
raw_labels=gmm_raw.predict(Z_raw)
robust=pd.DataFrame([
    {"Specification":"Log GNI + log1p CO2","AIC":gmm.aic(Z),"BIC":gmm.bic(Z),"Silhouette":silhouette_score(Z,base_labels)},
    {"Specification":"Raw GNI + raw CO2","AIC":gmm_raw.aic(Z_raw),"BIC":gmm_raw.bic(Z_raw),"Silhouette":silhouette_score(Z_raw,raw_labels)}
])
print("DÖNÜŞÜM SAĞLAMLIK KARŞILAŞTIRMASI:\n",robust.round(4).to_string(index=False))
print("\nARI (log vs raw):",round(adjusted_rand_score(base_labels,raw_labels),4))
print("NMI (log vs raw):",round(normalized_mutual_info_score(base_labels,raw_labels),4))

# 15. Rejimlerin ülke bazında listelenmesi ve HDI profilleri
Bu hücre her ülkenin GMM rejimini, üyelik güvenini ve normalize entropisini listeler. Ayrıca rejimlerin ortalama göstergelerini verir. Rejimlere sonradan anlamlı isimler verirken yalnızca bu profillere dayanmalısın; örneğin “yüksek gelişme–yüksek çevresel baskı” gibi isimler ancak ortalama değerler gerçekten destekliyorsa kullanılmalı.

In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from matplotlib.lines import Line2D

plot_df=df.reset_index(drop=True).copy()
assert len(plot_df)==len(Z),"df ve Z satır sayıları eşleşmiyor."

# GMM ETİKETLERİNİ BUL VEYA YENİDEN OLUŞTUR
label_candidates=["GMM_Regime","GMM_Cluster","GMM_Labels","gmm_labels","gmm_cluster","Cluster_GMM"]
label_col=next((c for c in label_candidates if c in plot_df.columns),None)

if label_col is not None:
    labels=plot_df[label_col].astype(int).reset_index(drop=True)
elif "gmm" in globals() and hasattr(gmm,"predict"):
    labels=pd.Series(gmm.predict(Z),name="GMM_Regime")
elif "gmm_final" in globals() and hasattr(gmm_final,"predict"):
    labels=pd.Series(gmm_final.predict(Z),name="GMM_Regime")
else:
    K=4
    gmm_model=GaussianMixture(n_components=K,covariance_type="full",n_init=20,random_state=42,reg_covar=1e-5)
    labels=pd.Series(gmm_model.fit_predict(Z),name="GMM_Regime")
    plot_df["GMM_Confidence"]=gmm_model.predict_proba(Z).max(axis=1)
    plot_df["GMM_Entropy"]=-(gmm_model.predict_proba(Z)*np.log(np.clip(gmm_model.predict_proba(Z),1e-12,1))).sum(axis=1)

plot_df["GMM_Regime"]=labels.to_numpy()
regimes=sorted(labels.unique())
colors=["#D73027","#2878B5","#238B45","#984EA3","#FF7F00","#00A6A6"]
color_map={r:colors[i%len(colors)] for i,r in enumerate(regimes)}

# T-SNE
perp=min(30,max(5,(len(plot_df)-1)//3))
coords=TSNE(n_components=2,perplexity=perp,learning_rate="auto",init="pca",max_iter=1500,random_state=42).fit_transform(Z)
plot_df["tSNE1"],plot_df["tSNE2"]=coords[:,0],coords[:,1]

fig,ax=plt.subplots(figsize=(17,13),facecolor="white")
ax.set_facecolor("white")
for r in regimes:
    sub=plot_df[plot_df["GMM_Regime"]==r]
    ax.scatter(sub["tSNE1"],sub["tSNE2"],s=105,c=color_map[r],edgecolors="black",linewidths=.7,alpha=.85,zorder=2)
    for _,row in sub.iterrows():
        country=row.get("Country",row.get("country",None))
        if pd.notna(country):
            ax.annotate(str(country),(row["tSNE1"],row["tSNE2"]),xytext=(4,4),textcoords="offset points",fontsize=8,fontweight="bold",color=color_map[r],zorder=3)

handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor=color_map[r],markeredgecolor="black",markersize=11,linewidth=0,label=f"Regime {r} (n={int((labels==r).sum())})") for r in regimes]
ax.legend(handles=handles,loc="upper right",frameon=True,facecolor="white",edgecolor="black",fontsize=12,title="GMM Socioeconomic Regimes",title_fontsize=13)
ax.set_title("t-SNE Visualization of GMM-Based Country Regimes",fontsize=19,fontweight="bold",pad=20)
ax.set_xlabel("t-SNE Dimension 1",fontsize=14,fontweight="bold")
ax.set_ylabel("t-SNE Dimension 2",fontsize=14,fontweight="bold")
ax.tick_params(axis="both",labelsize=10)
ax.grid(alpha=.18,linestyle="--")
for spine in ax.spines.values():spine.set_linewidth(1.2)
fig.tight_layout()
fig.savefig("GMM_tSNE_Country_Regimes.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

# ÜLKE VE REJİM SONUÇLARI
print("REJİM BAZINDA ÜLKE SAYILARI")
print(plot_df["GMM_Regime"].value_counts().sort_index().to_string())

country_col="Country" if "Country" in plot_df.columns else "country"
summary_cols=[c for c in ["HDI"]+list(features) if c in plot_df.columns]
confidence_cols=[c for c in ["GMM_Confidence","GMM_Entropy","FCM_Regime","FCM_Confidence"] if c in plot_df.columns]
country_results=plot_df[[country_col]+summary_cols+["GMM_Regime"]+confidence_cols].copy()
if "GMM_Confidence" in country_results.columns:
    country_results=country_results.sort_values(["GMM_Regime","GMM_Confidence"],ascending=[True,False])
else:
    country_results=country_results.sort_values("GMM_Regime")
print("\nÜLKE BAZINDA GMM SONUÇLARI")
display(country_results)
print("\nGMM REJİM ÖZETLERİ")
display(plot_df.groupby("GMM_Regime")[summary_cols].agg(["mean","median","std"]).round(3))


In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from matplotlib.lines import Line2D
from adjustText import adjust_text

plot_df=df.reset_index(drop=True).copy()
assert len(plot_df)==len(Z),"df ve Z satır sayıları eşleşmiyor."

label_candidates=["GMM_Regime","GMM_Cluster","GMM_Labels","gmm_labels","gmm_cluster","Cluster_GMM"]
label_col=next((c for c in label_candidates if c in plot_df.columns),None)

if label_col is not None:
    labels=plot_df[label_col].astype(int).reset_index(drop=True)
elif "gmm" in globals() and hasattr(gmm,"predict"):
    labels=pd.Series(gmm.predict(Z),name="GMM_Regime")
elif "gmm_final" in globals() and hasattr(gmm_final,"predict"):
    labels=pd.Series(gmm_final.predict(Z),name="GMM_Regime")
else:
    K=4
    gmm_model=GaussianMixture(n_components=K,covariance_type="full",n_init=20,random_state=42,reg_covar=1e-5)
    labels=pd.Series(gmm_model.fit_predict(Z),name="GMM_Regime")
    probs=gmm_model.predict_proba(Z)
    plot_df["GMM_Confidence"]=probs.max(axis=1)
    plot_df["GMM_Entropy"]=-(probs*np.log(np.clip(probs,1e-12,1))).sum(axis=1)

plot_df["GMM_Regime"]=labels.to_numpy()
regimes=sorted(labels.unique())
colors=["#D73027","#2878B5","#238B45","#984EA3","#FF7F00","#00A6A6"]
color_map={r:colors[i%len(colors)] for i,r in enumerate(regimes)}

country_col="Country" if "Country" in plot_df.columns else "country"
assert country_col in plot_df.columns,"Ülke isimleri için Country veya country sütunu bulunamadı."
assert plot_df[country_col].notna().all(),"Bazı ülke isimleri eksik."
assert len(labels)==len(Z),"GMM etiketleri ve Z satır sayıları eşleşmiyor."

perp=min(30,max(5,(len(plot_df)-1)//3))
coords=TSNE(n_components=2,perplexity=perp,learning_rate="auto",init="pca",max_iter=1500,random_state=42).fit_transform(Z)
plot_df["tSNE1"],plot_df["tSNE2"]=coords[:,0],coords[:,1]

fig,ax=plt.subplots(figsize=(24,19),facecolor="white")
ax.set_facecolor("#FAFBFD")
texts=[]

for r in regimes:
    sub=plot_df[plot_df["GMM_Regime"]==r]
    ax.scatter(sub["tSNE1"],sub["tSNE2"],s=85,c=color_map[r],edgecolors="black",linewidths=.6,alpha=.82,zorder=2)
    for _,row in sub.iterrows():
        texts.append(ax.text(row["tSNE1"],row["tSNE2"],str(row[country_col]),fontsize=7.5,fontweight="bold",color=color_map[r],zorder=3))

handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor=color_map[r],markeredgecolor="black",markersize=11,linewidth=0,label=f"Regime {r} (n={(labels==r).sum()})") for r in regimes]
ax.legend(handles=handles,loc="upper right",frameon=True,facecolor="white",edgecolor="black",fontsize=12,title="GMM Socioeconomic Regimes",title_fontsize=13)
ax.set_title("t-SNE Visualization of GMM-Based Country Regimes",fontsize=20,fontweight="bold",pad=22)
ax.set_xlabel("t-SNE Dimension 1",fontsize=14,fontweight="bold")
ax.set_ylabel("t-SNE Dimension 2",fontsize=14,fontweight="bold")
ax.tick_params(labelsize=10)
ax.grid(alpha=.18,linestyle="--")
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

fig.tight_layout()
adjust_text(texts,ax=ax,expand=(1.15,1.3),force_text=(.5,.8),force_static=(.3,.5),force_pull=(.02,.02),iter_lim=1000,arrowprops=dict(arrowstyle="-",color="#777777",lw=.45,alpha=.65))
fig.savefig("GMM_tSNE_Country_Regimes.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

print("REJİM BAZINDA ÜLKE SAYILARI")
print(plot_df["GMM_Regime"].value_counts().sort_index().to_string())

summary_cols=[c for c in ["HDI"]+list(features) if c in plot_df.columns]
confidence_cols=[c for c in ["GMM_Confidence","GMM_Entropy","FCM_Regime","FCM_Confidence"] if c in plot_df.columns]

country_results=plot_df[[country_col]+summary_cols+["GMM_Regime"]+confidence_cols].copy()
country_results=country_results.sort_values(["GMM_Regime",country_col])

print("\nÜLKE BAZINDA GMM SONUÇLARI - TÜM ÜLKELER")
print("Toplam ülke:",len(country_results))
with pd.option_context("display.max_rows",None,"display.max_columns",None,"display.width",None):
    display(country_results.reset_index(drop=True))

print("\nGMM REJİM ÖZETLERİ")
display(plot_df.groupby("GMM_Regime")[summary_cols].agg(["mean","median","std"]).round(3))


In [ ]:

fig,ax=plt.subplots(figsize=(25,20),facecolor="white")
ax.set_facecolor("#FFFFFF")
texts=[]

label_colors=["#A51C18","#174A73","#176B3A","#663399","#A34D00","#087A7A"]
label_map={r:label_colors[i%len(label_colors)] for i,r in enumerate(regimes)}

for r in regimes:
    sub=plot_df[plot_df["GMM_Regime"]==r]
    ax.scatter(sub["tSNE1"],sub["tSNE2"],s=100,c=color_map[r],edgecolors="black",linewidths=.7,alpha=.9,zorder=2)
    for _,row in sub.iterrows():
        texts.append(ax.text(row["tSNE1"],row["tSNE2"],str(row[country_col]),fontsize=9.5,fontweight="bold",color=label_map[r],ha="center",va="center",zorder=4,bbox=dict(facecolor="white",edgecolor="none",alpha=.78,pad=.8)))

handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor=color_map[r],markeredgecolor="black",markersize=12,linewidth=0,label=f"Regime {r} (n={(labels==r).sum()})") for r in regimes]

ax.legend(handles=handles,loc="upper right",frameon=True,facecolor="white",edgecolor="#777777",fontsize=13,title="GMM Socioeconomic Regimes",title_fontsize=14)
ax.set_title("t-SNE Visualization of GMM-Based Country Regimes",fontsize=22,fontweight="bold",pad=25)
ax.set_xlabel("t-SNE Dimension 1",fontsize=16,fontweight="bold")
ax.set_ylabel("t-SNE Dimension 2",fontsize=16,fontweight="bold")
ax.tick_params(axis="both",labelsize=12)
ax.grid(alpha=.15,linestyle="--")
ax.margins(.15)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

fig.tight_layout()

adjust_text(
    texts,
    ax=ax,
    expand=(1.3,1.5),
    force_text=(1.0,1.2),
    force_static=(.7,1.0),
    force_pull=(.01,.01),
    max_move=(50,50),
    iter_lim=2000,
    arrowprops=dict(arrowstyle="-",color="#777777",lw=.55,alpha=.65)
)

fig.savefig("GMM_tSNE_Country_Regimes.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()


In [ ]:

import numpy as np,pandas as pd,os

country_col=next((c for c in ["Country","country","COUNTRY"] if c in df.columns),None)
if country_col is None:raise KeyError(f"Ülke sütunu bulunamadı: {df.columns.tolist()}")
countries=df[country_col].reset_index(drop=True)
n=len(df)

for name,obj in {"gmm_prob":gmm_prob,"U":U,"A":A}.items():
    if len(obj)!=n:raise ValueError(f"{name}: satır sayısı df ile uyuşmuyor.")
    if np.asarray(obj).shape[1]!=best_k:raise ValueError(f"{name}: sütun sayısı best_k ile uyuşmuyor.")

prob_table=pd.DataFrame(gmm_prob,columns=[f"GMM_Probability_{i+1}" for i in range(best_k)])
prob_table.insert(0,"Country",countries)
fcm_table=pd.DataFrame(U,columns=[f"FCM_Membership_{i+1}" for i in range(best_k)])
fcm_table.insert(0,"Country",countries)
aa_table=pd.DataFrame(A,columns=[f"Archetype_Weight_{i+1}" for i in range(best_k)])
aa_table.insert(0,"Country",countries)
bootstrap_table=pd.DataFrame({"Bootstrap_ARI":np.asarray(ari_values).ravel()})

# MEVCUT DEĞİŞKENLERİ OTOMATİK TOPLA
candidate_tables={
"Country_Data_Results":"df",
"Descriptive_Statistics":"desc",
"Distribution":"dist",
"Correlation":"corr",
"PCA_Results":"pca_table",
"PCA_Loadings":"loadings",
"GMM_Selection":"gmm_table",
"GMM_Profiles":"profile",
"HDI_By_Regime":"hdi_profile",
"FCM_Profiles":"fcm_profile",
"Network_Centrality":"centrality",
"Robustness":"robust",
"Country_Classification":"country_results",
"GMM_Probabilities":"prob_table",
"FCM_Memberships":"fcm_table",
"Archetype_Weights":"aa_table",
"Bootstrap_ARI":"bootstrap_table"
}
tables={}
missing=[]
for sheet,var_name in candidate_tables.items():
    if var_name in globals():
        obj=globals()[var_name]
        if isinstance(obj,(pd.DataFrame,pd.Series)):
            tables[sheet]=obj.to_frame() if isinstance(obj,pd.Series) else obj
        else:missing.append(var_name)
    else:missing.append(var_name)

output_file="Human_Development_Archetypes_Results.xlsx"
with pd.ExcelWriter(output_file,engine="openpyxl") as writer:
    for sheet,table in tables.items():
        table.to_excel(writer,sheet_name=sheet[:31],index=sheet not in ["Country_Data_Results","PCA_Results","GMM_Selection","Robustness","Country_Classification","GMM_Probabilities","FCM_Memberships","Archetype_Weights","Bootstrap_ARI"])

print("Excel aktarımı tamamlandı.")
print("Dosya:",os.path.abspath(output_file))
print("Aktarılan sayfalar:",len(tables))
print("Aktarılan tablolar:",", ".join(tables.keys()))
print("Bulunmayan tablolar:",", ".join(missing) if missing else "Yok")


In [ ]:

import numpy as np,pandas as pd,os

# ÜLKE SÜTUNUNU BUL
country_col=next((c for c in df.columns if str(c).strip().lower() in ["country","countries","ülke"]),None)
if country_col is None:raise KeyError(f"Ülke sütunu bulunamadı. Sütunlar: {df.columns.tolist()}")

countries=df[country_col].to_numpy()
n=len(df)

# BOYUT KONTROLLERİ
for name,obj in {"gmm_prob":gmm_prob,"U":U,"A":A}.items():
    if np.asarray(obj).ndim!=2 or len(obj)!=n:
        raise ValueError(f"{name} boyutları df ile uyumlu değil: {np.asarray(obj).shape}")
    if np.asarray(obj).shape[1]!=best_k:
        raise ValueError(f"{name} sütun sayısı best_k ile uyumlu değil.")

# ÜLKE BAZLI TABLOLAR
prob_table=pd.DataFrame(gmm_prob,columns=[f"GMM_Probability_{i+1}" for i in range(best_k)])
prob_table.insert(0,"Country",countries)
fcm_table=pd.DataFrame(U,columns=[f"FCM_Membership_{i+1}" for i in range(best_k)])
fcm_table.insert(0,"Country",countries)
aa_table=pd.DataFrame(A,columns=[f"Archetype_Weight_{i+1}" for i in range(best_k)])
aa_table.insert(0,"Country",countries)
bootstrap_table=pd.DataFrame({"Bootstrap_ARI":np.asarray(ari_values).ravel()})

# EXCEL SAYFALARI: SADECE MEVCUT TABLOLAR
tables={
"Country_Data_Results":df,
"Descriptive_Statistics":globals().get("desc"),
"Distribution":globals().get("dist"),
"Correlation":globals().get("corr"),
"PCA_Results":globals().get("pca_table"),
"PCA_Loadings":globals().get("loadings"),
"GMM_Selection":globals().get("gmm_table"),
"GMM_Profiles":globals().get("profile"),
"HDI_By_Regime":globals().get("hdi_profile"),
"FCM_Profiles":globals().get("fcm_profile"),
"Network_Centrality":globals().get("centrality"),
"Robustness":globals().get("robust"),
"Country_Classification":globals().get("country_results"),
"GMM_Probabilities":prob_table,
"FCM_Memberships":fcm_table,
"Archetype_Weights":aa_table,
"Bootstrap_ARI":bootstrap_table
}
tables={k:(v.to_frame() if isinstance(v,pd.Series) else v) for k,v in tables.items() if isinstance(v,(pd.DataFrame,pd.Series))}

output_file="Human_Development_Archetypes_Results.xlsx"
with pd.ExcelWriter(output_file,engine="openpyxl") as writer:
    for sheet,table in tables.items():
        table.to_excel(writer,sheet_name=sheet[:31],index=sheet not in ["Country_Data_Results","PCA_Results","GMM_Selection","Robustness","Country_Classification","GMM_Probabilities","FCM_Memberships","Archetype_Weights","Bootstrap_ARI"])

print("Excel aktarımı tamamlandı.")
print("Dosya:",os.path.abspath(output_file))
print("Ülke sütunu:",country_col)
print("Gözlem sayısı:",n)
print("GMM bileşen sayısı:",best_k)
print("Aktarılan sayfalar:",list(tables.keys()))
print("Eksik tablolar:",[k for k,v in {
"Descriptive_Statistics":"desc","Distribution":"dist","Correlation":"corr","PCA_Results":"pca_table","PCA_Loadings":"loadings","GMM_Selection":"gmm_table","GMM_Profiles":"profile","HDI_By_Regime":"hdi_profile","FCM_Profiles":"fcm_profile","Network_Centrality":"centrality","Robustness":"robust","Country_Classification":"country_results"
}.items() if v not in globals()])


In [ ]:

import numpy as np,pandas as pd,os

# ÜLKE DEĞİŞKENİNİ BUL
country_col=next((c for c in df.columns if str(c).strip().lower() in ["country","countries","ülke"]),None)
if country_col is None:
    raise KeyError(f"Ülke sütunu bulunamadı. Mevcut sütunlar: {df.columns.tolist()}")

countries=df[country_col].to_numpy()
n=len(df)

# GMM OLASILIKLARI
gmm_prob=gmm.predict_proba(Z)
if len(gmm_prob)!=n:raise ValueError("GMM olasılıkları ile df satır sayısı eşleşmiyor.")
prob_table=pd.DataFrame(gmm_prob,columns=[f"GMM_Probability_{i+1}" for i in range(gmm_prob.shape[1])])
prob_table.insert(0,"Country",countries)

# FCM ÜYELİKLERİ
if len(U)!=n:raise ValueError("FCM üyelikleri ile df satır sayısı eşleşmiyor.")
fcm_table=pd.DataFrame(U,columns=[f"FCM_Membership_{i+1}" for i in range(U.shape[1])])
fcm_table.insert(0,"Country",countries)

# ARCHETYPAL ANALYSIS AĞIRLIKLARI
if len(A)!=n:raise ValueError("Archetype ağırlıkları ile df satır sayısı eşleşmiyor.")
aa_table=pd.DataFrame(A,columns=[f"Archetype_Weight_{i+1}" for i in range(A.shape[1])])
aa_table.insert(0,"Country",countries)

# BOOTSTRAP
bootstrap_table=pd.DataFrame({"Bootstrap_ARI":np.asarray(ari_values).ravel()})

# MEVCUT SONUÇ TABLOLARI
tables={
"Country_Data_Results":df,
"Descriptive_Statistics":desc,
"Distribution":dist,
"Correlation":corr,
"PCA_Results":pca_table,
"PCA_Loadings":loadings,
"GMM_Selection":gmm_table,
"GMM_Profiles":profile,
"HDI_By_Regime":hdi_profile,
"FCM_Profiles":fcm_profile,
"Robustness":robust,
"Country_Classification":country_results,
"GMM_Probabilities":prob_table,
"FCM_Memberships":fcm_table,
"Archetype_Weights":aa_table,
"Bootstrap_ARI":bootstrap_table
}

# NETWORK CENTRALITY VARSA EKLE
if "centrality" in globals():
    tables["Network_Centrality"]=centrality

# EXCEL
output_file="Human_Development_Archetypes_Results.xlsx"
with pd.ExcelWriter(output_file,engine="openpyxl") as writer:
    for sheet,table in tables.items():
        if isinstance(table,pd.Series):table=table.to_frame()
        table.to_excel(writer,sheet_name=sheet[:31],index=sheet not in ["Country_Data_Results","PCA_Results","GMM_Selection","Robustness","Country_Classification","GMM_Probabilities","FCM_Memberships","Archetype_Weights","Bootstrap_ARI"])

print("Excel dosyası başarıyla oluşturuldu.")
print("Dosya:",os.path.abspath(output_file))
print("Kullanılan ülke sütunu:",country_col)
print("Ülke sayısı:",n)
print("GMM bileşen sayısı:",gmm_prob.shape[1])
print("Aktarılan sayfa sayısı:",len(tables))
print("Sayfalar:",list(tables.keys()))


In [ ]:

import numpy as np,pandas as pd,matplotlib.pyplot as plt,glob,os
from IPython.display import display,Image

# 1. Ülke sütununu otomatik belirle
country_col=next((c for c in df.columns if str(c).strip().lower() in ["country","countries","ülke"]),None)
if country_col is None:
    raise KeyError(f"Ülke sütunu bulunamadı. Mevcut sütunlar: {df.columns.tolist()}")
countries=df[country_col].to_numpy()
n=len(df)

# 2. GMM olasılıkları
gmm_prob=gmm.predict_proba(Z)
if len(gmm_prob)!=n: raise ValueError("GMM ve df satır sayıları eşleşmiyor.")
prob_table=pd.DataFrame(gmm_prob,columns=[f"GMM_Probability_{i+1}" for i in range(gmm_prob.shape[1])])
prob_table.insert(0,"Country",countries)

# 3. FCM üyelikleri
if len(U)!=n: raise ValueError("FCM ve df satır sayıları eşleşmiyor.")
fcm_table=pd.DataFrame(U,columns=[f"FCM_Membership_{i+1}" for i in range(U.shape[1])])
fcm_table.insert(0,"Country",countries)

# 4. Archetypal Analysis ağırlıkları
if len(A)!=n: raise ValueError("Archetypal Analysis ve df satır sayıları eşleşmiyor.")
aa_table=pd.DataFrame(A,columns=[f"Archetype_Weight_{i+1}" for i in range(A.shape[1])])
aa_table.insert(0,"Country",countries)

# 5. Bootstrap sonuçları
bootstrap_table=pd.DataFrame({"Bootstrap_ARI":np.asarray(ari_values).ravel()})

# 6. Mevcut tabloları otomatik topla
candidate_tables={
"Country_Data_Results":"df","Descriptive_Statistics":"desc","Distribution":"dist",
"Correlation":"corr","PCA_Results":"pca_table","PCA_Loadings":"loadings",
"GMM_Selection":"gmm_table","GMM_Profiles":"profile","HDI_By_Regime":"hdi_profile",
"FCM_Profiles":"fcm_profile","Network_Centrality":"centrality","Robustness":"robust",
"Country_Classification":"country_results","GMM_Probabilities":"prob_table",
"FCM_Memberships":"fcm_table","Archetype_Weights":"aa_table","Bootstrap_ARI":"bootstrap_table"
}
tables={}
for sheet,var in candidate_tables.items():
    obj=globals().get(var)
    if isinstance(obj,pd.Series): obj=obj.to_frame()
    if isinstance(obj,pd.DataFrame): tables[sheet]=obj

# 7. Excel dosyasını oluştur
output_file="Human_Development_Archetypes_Results.xlsx"
with pd.ExcelWriter(output_file,engine="openpyxl") as writer:
    for sheet,table in tables.items():
        table.to_excel(writer,sheet_name=sheet,index=(sheet not in ["Country_Data_Results","PCA_Results","GMM_Selection","Robustness","Country_Classification","GMM_Probabilities","FCM_Memberships","Archetype_Weights","Bootstrap_ARI"]))

print("EXCEL DOSYASI:",os.path.abspath(output_file))
print("OLUŞTURULAN SAYFA SAYISI:",len(tables))

# 8. Bütün tabloları Notebook'ta göster
display_names={
"Country_Data_Results":"ÜLKE BAZINDA HAM VERİ VE SONUÇLAR",
"Descriptive_Statistics":"TANIMLAYICI İSTATİSTİKLER",
"Distribution":"DEĞİŞKEN DAĞILIMLARI",
"Correlation":"KORELASYON MATRİSİ",
"PCA_Results":"PCA SONUÇLARI","PCA_Loadings":"PCA YÜKLERİ",
"GMM_Selection":"GMM MODEL SEÇİMİ","GMM_Profiles":"GMM REJİM PROFİLLERİ",
"HDI_By_Regime":"HDI REJİM KARŞILAŞTIRMASI","FCM_Profiles":"FCM REJİM PROFİLLERİ",
"Network_Centrality":"AĞ MERKEZİYET ANALİZİ","Robustness":"ROBUSTLUK ANALİZİ",
"Country_Classification":"ÜLKE SINIFLANDIRMALARI","GMM_Probabilities":"GMM OLASILIKLARI",
"FCM_Memberships":"FCM ÜYELİK DEĞERLERİ","Archetype_Weights":"ARCHETYPAL ANALYSIS AĞIRLIKLARI",
"Bootstrap_ARI":"GMM BOOTSTRAP ARI SONUÇLARI"
}
with pd.option_context("display.max_rows",None,"display.max_columns",None,"display.width",None):
    for sheet,table in tables.items():
        print("\n"+"="*100+"\n"+display_names.get(sheet,sheet)+"\n"+"="*100)
        display(table.round(4))

# 9. Mevcut grafikleri göster
figure_files=sorted(set(glob.glob("[0-9][0-9]_*.png")+glob.glob("[0-9][0-9][0-9]_*.png")+glob.glob("GMM_tSNE*.png")))
print("\n"+"="*100+"\nGRAFİKLER\n"+"="*100)
for file in figure_files:
    print("GRAFİK:",file)
    display(Image(filename=file))
if not figure_files: print("Belirtilen isimlendirmeyle PNG grafik bulunamadı.")

# 10. GMM rejim dağılımı
if "GMM_Regime" in df.columns:
    regime_counts=df["GMM_Regime"].value_counts().sort_index()
    regime_summary=pd.DataFrame({"Country_Count":regime_counts,"Percentage":(regime_counts/n*100).round(2)})
    print("\nGMM REJİM DAĞILIMI")
    display(regime_summary)
else:
    print("\nUYARI: df içinde GMM_Regime sütunu bulunamadı.")

print("\nTÜM MEVCUT TABLOLAR GÖSTERİLDİ.")
print("EXCEL:",os.path.abspath(output_file))
print("GÖSTERİLEN GRAFİK SAYISI:",len(figure_files))


# 18. SHAP ve CatBoost kütüphanelerinin kurulması

In [ ]:
import shap
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, log_loss
print("SHAP:",shap.__version__)
print("CatBoost:",__import__("catboost").__version__)
X_shap=Xdf.copy()
y_shap=np.asarray(gmm_labels,dtype=int)
class_counts=pd.Series(y_shap+1).value_counts().sort_index()
print("GMM REJİM DAĞILIMI:\n",class_counts.to_string())
n_splits=min(5,int(class_counts.min()))
if n_splits<2: raise ValueError("En az iki gözlem içeren her rejim gerekli; GMM küme sayısını veya model tasarımını gözden geçir.")
cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
cat_params={"iterations":500,"depth":5,"learning_rate":0.03,"loss_function":"MultiClass","eval_metric":"MultiClass","random_seed":SEED,"verbose":False,"allow_writing_files":False,"l2_leaf_reg":5}
print(f"\nCatBoost sınıflandırması hazır. Stratified CV: {n_splits} kat.")


In [ ]:
cv_model=CatBoostClassifier(**cat_params)
cv_pred=cross_val_predict(cv_model,X_shap,y_shap,cv=cv,method="predict",n_jobs=1).ravel().astype(int)
cv_prob=cross_val_predict(cv_model,X_shap,y_shap,cv=cv,method="predict_proba",n_jobs=1)
metrics={"Accuracy":accuracy_score(y_shap,cv_pred),"Balanced accuracy":balanced_accuracy_score(y_shap,cv_pred),"Macro-F1":f1_score(y_shap,cv_pred,average="macro"),"Weighted-F1":f1_score(y_shap,cv_pred,average="weighted"),"Log loss":log_loss(y_shap,cv_prob,labels=np.arange(best_k))}
print("STRATIFIED CROSS-VALIDATION SONUÇLARI")
for k,v in metrics.items(): print(f"{k}: {v:.4f}")
print("\nSINIFLANDIRMA RAPORU:\n",classification_report(y_shap,cv_pred,labels=np.arange(best_k),target_names=[f"Regime {i}" for i in range(1,best_k+1)],zero_division=0))
cm=confusion_matrix(y_shap,cv_pred,labels=np.arange(best_k),normalize="true")
fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm,cmap="YlGnBu",vmin=0,vmax=1)
ax.set_xticks(range(best_k),[f"R{i}" for i in range(1,best_k+1)],fontweight="bold")
ax.set_yticks(range(best_k),[f"R{i}" for i in range(1,best_k+1)],fontweight="bold")
ax.set_xlabel("Predicted regime"); ax.set_ylabel("Actual GMM regime")
ax.set_title("CatBoost Cross-Validated Confusion Matrix",color="black")
for i in range(best_k):
    for j in range(best_k): ax.text(j,i,f"{cm[i,j]:.2f}",ha="center",va="center",color="black",fontweight="bold",fontsize=12)
fig.colorbar(im,ax=ax,label="Row-normalized proportion")
finish(fig,"14_shap_model_confusion_matrix.png")

In [ ]:
import numpy as np
import shap
from catboost import CatBoostClassifier

shap_model=CatBoostClassifier(**cat_params)
shap_model.fit(X_shap,y_shap)
explainer=shap.TreeExplainer(shap_model)
sv_raw=explainer.shap_values(X_shap)

n,p=X_shap.shape
k=len(shap_model.classes_)

if isinstance(sv_raw,list):
    sv=np.stack([np.asarray(v) for v in sv_raw],axis=-1)
else:
    sv=np.asarray(sv_raw)
    if sv.ndim==2:
        sv=sv[:,:,None]
    elif sv.ndim==3:
        if sv.shape==(n,p,k):
            pass
        elif sv.shape==(k,n,p):
            sv=np.transpose(sv,(1,2,0))
        elif sv.shape==(n,k,p):
            sv=np.transpose(sv,(0,2,1))
        elif sv.shape==(n,p,k+1):
            sv=sv[:,:,:k]
        else:
            raise ValueError(f"Tanımlanamayan SHAP boyutu: {sv.shape}")

if sv.shape[:2]!=(n,p):
    raise ValueError(f"SHAP boyutu uyumsuz: {sv.shape}; beklenen ilk iki boyut: {(n,p)}")

feature_names=list(X_shap.columns) if hasattr(X_shap,"columns") else list(features[:p])

if len(feature_names)!=p:
    raise ValueError("Değişken isimleri ile X_shap sütun sayısı eşleşmiyor.")

print("SHAP boyutu:",sv.shape)
print("Gözlem sayısı:",n)
print("Değişken sayısı:",p)
print("Açıklanan sınıf sayısı:",sv.shape[2])
print("Değişkenler:",feature_names)
print("GMM sınıfları:",shap_model.classes_)
print("SHAP hesaplaması başarıyla tamamlandı.")

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt

feature_names=list(X_shap.columns)
global_imp=np.abs(sv).mean(axis=(0,2))

if len(feature_names)!=len(global_imp):
    raise ValueError(f"Değişken sayısı ({len(feature_names)}) ile SHAP boyutu ({len(global_imp)}) eşleşmiyor.")

shap_global=pd.DataFrame({"Variable":feature_names,"Mean absolute SHAP":global_imp}).sort_values("Mean absolute SHAP",ascending=True)

print("GLOBAL SHAP DEĞİŞKEN ÖNEMİ:")
print(shap_global.sort_values("Mean absolute SHAP",ascending=False).round(5).to_string(index=False))

fig,ax=plt.subplots(figsize=(10,7),facecolor="white")
ax.set_facecolor("white")
colors=plt.cm.Greens(np.linspace(.45,.90,len(shap_global)))
bars=ax.barh(shap_global["Variable"],shap_global["Mean absolute SHAP"],color=colors,edgecolor="black",linewidth=1.1,height=.65)

for bar,v in zip(bars,shap_global["Mean absolute SHAP"]):
    ax.text(v+global_imp.max()*.015,bar.get_y()+bar.get_height()/2,f"{v:.4f}",va="center",ha="left",fontsize=11,fontweight="bold",color="black")

ax.set_xlabel("Mean |SHAP value|",fontsize=13,fontweight="bold")
ax.set_ylabel("Indicator",fontsize=13,fontweight="bold")
ax.set_title("Global SHAP Feature Importance",fontsize=16,fontweight="bold",color="black",pad=16)
ax.set_xlim(0,global_imp.max()*1.18)
ax.tick_params(axis="both",labelsize=11)
ax.grid(axis="x",alpha=.20,linestyle="--")
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_linewidth(1.1)
fig.tight_layout()
finish(fig,"15_global_shap_importance.png")
plt.show()

In [ ]:
print("BAĞIMLI DEĞİŞKEN:",pd.Series(y_shap).name)
print("X_SHAP BOYUTU:",X_shap.shape)
print("SHAP BOYUTU:",sv.shape)
print("BAĞIMLI DEĞİŞKENDEKİ SINIFLAR:",np.unique(y_shap))
print("AÇIKLAYICI DEĞİŞKENLER:",list(X_shap.columns))
print("GMM REJİM DAĞILIMI:")
print(pd.Series(y_shap).value_counts().sort_index())
print("CATBOOST SINIFLARI:",shap_model.classes_)

SHAP ANALIZI

In [ ]:
import os,warnings,shap,numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold,cross_val_predict
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,HistGradientBoostingRegressor
from catboost import CatBoostRegressor
warnings.filterwarnings("ignore")
SEED=42
N_SPLITS=5
B=1000
OUT="HDI_SHAP_Results"
os.makedirs(OUT,exist_ok=True)
features=["CHE","CO2","EA"]
target="HDI"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":"black","axes.labelcolor":"black","text.color":"black","xtick.color":"black","ytick.color":"black","font.family":"Arial","font.weight":"bold","axes.labelweight":"bold","axes.titleweight":"bold","font.size":11,"axes.titlesize":14,"axes.labelsize":11,"xtick.labelsize":10,"ytick.labelsize":10,"savefig.dpi":600,"savefig.bbox":"tight"})
COLORS=["#00A878","#2878B5","#F28E2B","#8E63B5","#D64F4F","#17A2B8","#E3B505","#536872"]
def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{OUT}/{name}.png",dpi=600,bbox_inches="tight",facecolor="white")
    plt.savefig(f"{OUT}/{name}.pdf",bbox_inches="tight",facecolor="white")
    plt.show()
def show_table(data,title):
    print("\n"+title)
    display(data.style.format(precision=4).background_gradient(cmap="YlGnBu",axis=None).set_properties(**{"color":"black","font-weight":"bold"}).set_table_styles([{"selector":"th","props":[("color","black"),("font-weight","bold"),("background-color","#DCE6F1")]}]))
print("SHAP:",shap.__version__)
print("Analiz ayarları hazır.")

In [ ]:
df=pd.read_csv("HDI.csv",sep=";",encoding="ISO-8859-9")
df.columns=df.columns.str.strip()
if "country" not in df.columns and "Country" in df.columns:
    df=df.rename(columns={"Country":"country"})
required=["HDI","LE","EYS","MYS","GNI","CHE","CO2","EA"]
missing_cols=[c for c in required if c not in df.columns]
if missing_cols: raise ValueError(f"Eksik sütunlar: {missing_cols}. Mevcut sütunlar: {df.columns.tolist()}")
for c in required:
    if c=="GNI":
        df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","",regex=False),errors="coerce")
    else:
        df[c]=pd.to_numeric(df[c],errors="coerce")
print("Veri boyutu:",df.shape)
print("Değişkenler:",df.columns.tolist())
print("\nEksik gözlem sayıları:")
show_table(df[required].isna().sum().to_frame("Missing values"),"MISSING VALUE CHECK")
if "country" in df.columns:
    print("Tekrarlanan ülke sayısı:",df["country"].duplicated().sum())

In [ ]:
desc=df[required].describe().T
desc["Skewness"]=df[required].skew()
desc["Kurtosis"]=df[required].kurtosis()
show_table(desc,"DESCRIPTIVE STATISTICS OF ALL VARIABLES")

In [ ]:
corr=df[required].corr(method="pearson")
show_table(corr,"PEARSON CORRELATION MATRIX")
fig,ax=plt.subplots(figsize=(9,7))
im=ax.imshow(corr.values,cmap="RdYlBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(required)),required,fontweight="bold")
ax.set_yticks(range(len(required)),required,fontweight="bold")
ax.set_title("Correlation Matrix of Human Development Indicators",fontweight="bold")
for i in range(len(required)):
    for j in range(len(required)):
        ax.text(j,i,f"{corr.iloc[i,j]:.3f}",ha="center",va="center",color="black",fontweight="bold")
fig.colorbar(im,ax=ax,label="Pearson correlation")
savefig("Figure_01_Correlation_Matrix")
print(corr)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED=42
features=["CHE","CO2","EA"]
X=data[features].copy()
y=data["HDI"].copy()
valid=X.notna().all(axis=1)&y.notna()
X,y=X.loc[valid],y.loc[valid]
if len(X)<5: raise ValueError("5-fold CV için yeterli gözlem yok.")
if not np.isfinite(X.to_numpy(dtype=float)).all() or not np.isfinite(y.to_numpy(dtype=float)).all():
    raise ValueError("Veriler sonsuz değer içeriyor.")

models={
"KNN Regression":Pipeline([("scaler",StandardScaler()),("model",KNeighborsRegressor(n_neighbors=5,weights="distance",p=2))]),
"Elastic Net":Pipeline([("scaler",StandardScaler()),("model",ElasticNet(alpha=0.01,l1_ratio=0.5,max_iter=20000))]),
"SVR RBF":Pipeline([("scaler",StandardScaler()),("model",SVR(kernel="rbf",C=10,epsilon=0.01,gamma="scale"))]),
"Random Forest":RandomForestRegressor(n_estimators=500,max_features=1.0,min_samples_leaf=2,random_state=SEED,n_jobs=-1),
"Extra Trees":ExtraTreesRegressor(n_estimators=500,max_features=1.0,min_samples_leaf=2,random_state=SEED,n_jobs=-1),
"Gradient Boosting":GradientBoostingRegressor(n_estimators=300,learning_rate=0.03,max_depth=2,loss="huber",random_state=SEED),
"Hist Gradient Boosting":HistGradientBoostingRegressor(max_iter=300,learning_rate=0.04,max_leaf_nodes=7,l2_regularization=2,random_state=SEED),
"CatBoost":CatBoostRegressor(iterations=500,depth=4,learning_rate=0.03,loss_function="RMSE",l2_leaf_reg=5,random_seed=SEED,verbose=False,allow_writing_files=False),
"LightGBM":LGBMRegressor(n_estimators=500,max_depth=4,num_leaves=7,learning_rate=0.03,min_child_samples=5,reg_lambda=5,random_state=SEED,n_jobs=-1,verbosity=-1),
"XGBoost":XGBRegressor(n_estimators=500,max_depth=3,learning_rate=0.03,subsample=0.85,colsample_bytree=0.85,reg_lambda=5,objective="reg:squarederror",random_state=SEED,n_jobs=-1)
}

cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
oof_predictions={}
rows=[]
for name,model in models.items():
    pred=cross_val_predict(clone(model),X,y,cv=cv,n_jobs=1)
    mse=mean_squared_error(y,pred)
    oof_predictions[name]=pred
    rows.append({"Algorithm":name,"OOF_R2":r2_score(y,pred),"OOF_RMSE":np.sqrt(mse),"OOF_MAE":mean_absolute_error(y,pred),"OOF_MSE":mse})

results=pd.DataFrame(rows).sort_values("OOF_RMSE").reset_index(drop=True)
best_name=results.iloc[0]["Algorithm"]
best_model=clone(models[best_name]).fit(X,y)

metrics=[("OOF_R2","R²"),("OOF_RMSE","RMSE"),("OOF_MAE","MAE"),("OOF_MSE","MSE")]
palette=dict(zip(models,plt.cm.tab10(np.linspace(0,1,len(models)))))
plt.rcParams.update({"font.family":"DejaVu Sans","axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})

fig,axes=plt.subplots(2,2,figsize=(22,19),facecolor="white")
for ax,(metric,title) in zip(axes.flat,metrics):
    plot=results.sort_values(metric,ascending=(metric!="OOF_R2")).reset_index(drop=True)
    bars=ax.bar(range(len(plot)),plot[metric],color=[palette[n] for n in plot["Algorithm"]],edgecolor="black",linewidth=1)
    ax.set_title(title,fontsize=22,fontweight="bold",color="black")
    ax.set_ylabel(title,fontsize=22,fontweight="bold",color="black")
    ax.set_xticks(range(len(plot)))
    ax.set_xticklabels(plot["Algorithm"],rotation=35,ha="right",fontsize=18,fontweight="bold",color="black")
    ax.tick_params(axis="y",labelsize=18,colors="black")
    ax.grid(axis="y",alpha=.2,linestyle="--")
    ax.set_axisbelow(True)
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")
    vals=plot[metric].to_numpy()
    ymin=min(0,vals.min())
    ymax=max(0,vals.max())
    span=max(ymax-ymin,1e-9)
    ax.set_ylim(ymin-min(0,span*.08),ymax+span*.28)
    for bar,val in zip(bars,vals):
        ax.annotate(f"{val:.3f}",(bar.get_x()+bar.get_width()/2,val),xytext=(0,5 if val>=0 else -5),textcoords="offset points",ha="center",va="bottom" if val>=0 else "top",fontsize=16,fontweight="bold",color="black",rotation=90)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Comparative Performance of Ten Machine Learning Algorithms for HDI Prediction",fontsize=25,fontweight="bold",y=.995)
fig.tight_layout(rect=[0,0,1,.97])
fig.savefig("Figure_ML_Performance_HDI_CHE_CO2_EA.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

cols=["Algorithm","OOF_R2","OOF_RMSE","OOF_MAE","OOF_MSE"]
print("\nHDI PREDICTION USING CHE, CO2 AND EA")
print("Observations:",len(X))
print("Predictors:",features)
print(results[cols].to_string(index=False,float_format=lambda x:f"{x:.6f}"))
display(results[cols].style.format({c:"{:.6f}" for c in cols[1:]}).background_gradient(cmap="YlGnBu",subset=cols[1:]).set_properties(**{"color":"black","font-weight":"bold"}))
print("\nBEST PERFORMING ALGORITHM:",best_name)
print("BEST OOF R²:",f"{results.iloc[0]['OOF_R2']:.6f}")
print("BEST OOF RMSE:",f"{results.iloc[0]['OOF_RMSE']:.6f}")
results.to_csv("Table_ML_Performance_HDI_3_Predictors.csv",index=False)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

features=["CHE","CO2","EA"]
if list(X.columns)!=features:
    raise ValueError("X yalnızca CHE, CO2 ve EA değişkenlerini içermelidir. Önce üç değişkenli model kodunu çalıştırın.")

top9=results.nsmallest(9,"OOF_RMSE")["Algorithm"].tolist()
observed=np.asarray(y).ravel()
colors=plt.cm.tab10(np.linspace(0,1,9))
fig,axes=plt.subplots(3,3,figsize=(24,23),facecolor="white")
performance_rows=[]
all_preds=np.concatenate([np.asarray(oof_predictions[name]).ravel() for name in top9])
margin=max((max(observed.max(),all_preds.max())-min(observed.min(),all_preds.min()))*.04,1e-6)
lims=[min(observed.min(),all_preds.min())-margin,max(observed.max(),all_preds.max())+margin]

for i,(ax,name,color) in enumerate(zip(axes.flat,top9,colors),1):
    pred=np.asarray(oof_predictions[name]).ravel()
    if len(pred)!=len(observed):
        raise ValueError(f"{name}: Tahmin ve gözlem sayıları uyuşmuyor.")
    r2=r2_score(observed,pred)
    mse=mean_squared_error(observed,pred)
    rmse=np.sqrt(mse)
    mae=mean_absolute_error(observed,pred)
    bias=np.mean(pred-observed)
    max_error=np.max(np.abs(pred-observed))
    n=len(observed)
    performance_rows.append({"Algorithm":name,"N":n,"R2":r2,"MSE":mse,"RMSE":rmse,"MAE":mae,"Bias":bias,"Max Absolute Error":max_error})
    ax.scatter(observed,pred,s=55,c=[color],edgecolors="black",linewidths=.7,alpha=.85)
    ax.plot(lims,lims,"--",color="#D64F4F",linewidth=2.5,label="Perfect prediction")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal",adjustable="box")
    ax.set_xlabel("Observed HDI",fontsize=18,fontweight="bold",color="black")
    ax.set_ylabel("Out-of-fold Predicted HDI",fontsize=18,fontweight="bold",color="black")
    ax.set_title(f"({chr(96+i)}) {name}",fontsize=21,fontweight="bold",color="black",pad=15)
    ax.tick_params(axis="both",labelsize=17,colors="black")
    for label in ax.get_xticklabels()+ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")
    info=f"N = {n}\nR² = {r2:.3f}\nMSE = {mse:.4f}\nRMSE = {rmse:.3f}\nMAE = {mae:.3f}\nBias = {bias:.3f}"
    ax.text(.04,.96,info,transform=ax.transAxes,va="top",ha="left",fontsize=14,fontweight="bold",color="black",bbox=dict(facecolor="white",edgecolor="black",alpha=.95,boxstyle="round,pad=.4"))
    ax.legend(loc="lower right",fontsize=12,frameon=True)
    ax.grid(alpha=.15,linestyle="--")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Observed vs. Predicted HDI Using CHE, CO₂ and EA: Nine Machine Learning Models",fontsize=26,fontweight="bold",y=.995)
fig.tight_layout(rect=[0,0,1,.975],h_pad=3,w_pad=2)
fig.savefig("Figure_04_Observed_Predicted_HDI_CHE_CO2_EA.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

performance_9=pd.DataFrame(performance_rows).sort_values("RMSE").reset_index(drop=True)
print("\nHDI PREDICTION USING CHE, CO2 AND EA - NINE MODELS")
print("Predictors:",features)
print("Observations:",len(observed))
print(performance_9.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
display(performance_9.style.format({c:"{:.4f}" for c in performance_9.columns if c not in ["Algorithm","N"]}).background_gradient(cmap="YlGnBu",subset=["R2","MSE","RMSE","MAE"]).set_properties(**{"color":"black","font-weight":"bold"}))
performance_9.to_csv("Table_Observed_Predicted_HDI_3_Predictors.csv",index=False)
print("\nBEST PERFORMING ALGORITHM:",performance_9.iloc[0]["Algorithm"])
print("EXCLUDED ALGORITHM:",[name for name in results["Algorithm"] if name not in top9])


In [ ]:

top4=results.nsmallest(4,"OOF_RMSE")["Algorithm"].tolist()
observed=np.asarray(y).ravel()
colors=["#00A878","#2878B5","#E69F00","#8E63B6"]
fig,axes=plt.subplots(2,2,figsize=(18,17),facecolor="white")
performance_rows=[]
all_preds=np.concatenate([np.asarray(oof_predictions[name]).ravel() for name in top4])
margin=(max(observed.max(),all_preds.max())-min(observed.min(),all_preds.min()))*.04
lims=[min(observed.min(),all_preds.min())-margin,max(observed.max(),all_preds.max())+margin]

for ax,name,color in zip(axes.flat,top4,colors):
    pred=np.asarray(oof_predictions[name]).ravel()
    r2=r2_score(observed,pred)
    mse=mean_squared_error(observed,pred)
    rmse=np.sqrt(mse)
    mae=mean_absolute_error(observed,pred)
    bias=np.mean(pred-observed)
    max_error=np.max(np.abs(pred-observed))
    n=len(observed)
    performance_rows.append({"Algorithm":name,"N":n,"R2":r2,"MSE":mse,"RMSE":rmse,"MAE":mae,"Bias":bias,"Max Absolute Error":max_error})
    ax.scatter(observed,pred,s=65,c=color,edgecolors="black",linewidths=.7,alpha=.85)
    ax.plot(lims,lims,"--",color="#D64F4F",linewidth=2.5,label="Perfect prediction")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal",adjustable="box")
    ax.set_xlabel("Observed HDI",fontsize=19,fontweight="bold",color="black")
    ax.set_ylabel("Out-of-fold Predicted HDI",fontsize=19,fontweight="bold",color="black")
    ax.set_title(name,fontsize=22,fontweight="bold",color="black",pad=15)
    ax.tick_params(axis="both",labelsize=18,colors="black")
    for label in ax.get_xticklabels()+ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")
    info=f"N = {n}\nR² = {r2:.3f}\nMSE = {mse:.4f}\nRMSE = {rmse:.3f}\nMAE = {mae:.3f}\nBias = {bias:.3f}"
    ax.text(.04,.96,info,transform=ax.transAxes,va="top",ha="left",fontsize=15,fontweight="bold",color="black",bbox=dict(facecolor="white",edgecolor="black",alpha=.95,boxstyle="round,pad=.5"))
    ax.legend(loc="lower right",fontsize=14,frameon=True)
    ax.grid(alpha=.15,linestyle="--")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Observed vs. Predicted HDI: Top Four Machine Learning Models",fontsize=25,fontweight="bold",y=.995)
fig.tight_layout(rect=[0,0,1,.97],h_pad=3,w_pad=2)
fig.savefig("Figure_04_Top4_Observed_Predicted_HDI.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

top4_performance=pd.DataFrame(performance_rows).sort_values("RMSE").reset_index(drop=True)
print("\nTOP FOUR MODELS - OUT-OF-FOLD PERFORMANCE")
print(top4_performance.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
display(top4_performance.style.format({c:"{:.4f}" for c in top4_performance.columns if c not in ["Algorithm","N"]}).background_gradient(cmap="YlGnBu",subset=["R2","MSE","RMSE","MAE"]).set_properties(**{"color":"black","font-weight":"bold"}))


In [ ]:

best_oof=np.asarray(oof_predictions[best_name])
observed=np.asarray(y)
r2=r2_score(observed,best_oof)
mse=mean_squared_error(observed,best_oof)
rmse=np.sqrt(mse)
mae=mean_absolute_error(observed,best_oof)
bias=np.mean(best_oof-observed)
max_error=np.max(np.abs(best_oof-observed))
n=len(observed)

fig,ax=plt.subplots(figsize=(9,9),facecolor="white")
ax.scatter(observed,best_oof,s=75,c="#00A878",edgecolors="black",linewidths=0.8,alpha=0.85)
lims=[min(observed.min(),best_oof.min())-0.02,max(observed.max(),best_oof.max())+0.02]
ax.plot(lims,lims,"--",color="#D64F4F",linewidth=2.5,label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Observed HDI",fontsize=19,fontweight="bold",color="black")
ax.set_ylabel("Out-of-fold Predicted HDI",fontsize=19,fontweight="bold",color="black")
ax.set_title(f"Observed vs. Predicted HDI: {best_name}",fontsize=19,fontweight="bold",color="black")
ax.tick_params(axis="both",labelsize=16,colors="black")
for label in ax.get_xticklabels()+ax.get_yticklabels():
    label.set_fontweight("bold")
ax.legend(loc="lower right",fontsize=14,frameon=True)
ax.text(0.04,0.96,f"N = {n}\nR² = {r2:.4f}\nMSE = {mse:.6f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\nBias = {bias:.4f}",transform=ax.transAxes,va="top",fontsize=15,fontweight="bold",color="black",bbox={"facecolor":"white","edgecolor":"black","alpha":0.95})
ax.grid(alpha=0.15,linestyle="--")
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("Figure_04_Observed_Predicted_HDI.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

performance=pd.DataFrame({"Metric":["Algorithm","Observations (N)","R²","MSE","RMSE","MAE","Bias","Maximum Absolute Error"],"Value":[best_name,n,r2,mse,rmse,mae,bias,max_error]})
print("\nBEST MODEL - OUT-OF-FOLD PERFORMANCE")
print(f"Algorithm              : {best_name}")
print(f"Observations (N)       : {n}")
print(f"R²                     : {r2:.6f}")
print(f"MSE                    : {mse:.6f}")
print(f"RMSE                   : {rmse:.6f}")
print(f"MAE                    : {mae:.6f}")
print(f"Bias                   : {bias:.6f}")
print(f"Maximum Absolute Error : {max_error:.6f}")
display(performance)


In [ ]:
residuals=y.to_numpy()-best_oof
residual_table=pd.DataFrame({"Country":data["country"],"Observed_HDI":y,"Predicted_HDI_OOF":best_oof,"Residual":residuals,"Absolute_Error":np.abs(residuals)})
show_table(residual_table[["Residual","Absolute_Error"]].describe().T,"RESIDUAL STATISTICS")
fig,ax=plt.subplots(figsize=(9,5))
ax.scatter(best_oof,residuals,s=55,c="#2878B5",edgecolors="black",alpha=0.8)
ax.axhline(0,color="#D64F4F",linestyle="--",linewidth=2)
ax.set_xlabel("OOF predicted HDI",fontweight="bold")
ax.set_ylabel("Residual (observed − predicted)",fontweight="bold")
ax.set_title(f"Residual Diagnostic: {best_name}",fontweight="bold")
savefig("Figure_05_Residual_Diagnostic")

In [ ]:
background=X.sample(min(50,len(X)),random_state=SEED)
masker=shap.maskers.Independent(background)
explainer=shap.PermutationExplainer(best_model.predict,masker,seed=SEED)
shap_exp=explainer(X,max_evals=2*X.shape[1]+1)
sv=np.asarray(shap_exp.values)
if sv.shape!=X.shape: raise ValueError(f"SHAP boyutu beklenmeyen biçimde: {sv.shape}")
base_values=np.asarray(shap_exp.base_values).reshape(-1)
print("SHAP boyutu:",sv.shape)
print("SHAP algoritması:",best_name)
print("SHAP yöntemi: Permutation SHAP")
print("SHAP açıklanan değer: HDI tahmini")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.base import clone
from IPython.display import display

features=["CHE","CO2","EA"]
if not isinstance(X,pd.DataFrame) or not all(v in X.columns for v in features):
    raise ValueError("X, CHE, CO2 ve EA sütunlarını içermelidir.")
X3=X[features].copy()
if X3.isna().any().any() or not np.isfinite(X3.to_numpy(dtype=float)).all():
    raise ValueError("Bağımsız değişkenlerde eksik veya sonsuz değer var.")
if len(X3)!=len(y):
    raise ValueError("X ve y gözlem sayıları uyuşmuyor.")

best_model=clone(models[best_name]).fit(X3,y)
background=shap.sample(X3,min(60,len(X3)),random_state=42)
explainer=shap.Explainer(best_model.predict,background,algorithm="permutation",seed=42)
sv=explainer(X3,max_evals=2*len(features)+1)
S=np.asarray(sv.values)
if S.shape!=(len(X3),3):
    raise ValueError(f"Beklenen SHAP boyutu {(len(X3),3)}, elde edilen {S.shape}.")

global_shap=pd.DataFrame({
    "Variable":features,
    "Mean absolute SHAP":np.abs(S).mean(axis=0),
    "Mean signed SHAP":S.mean(axis=0),
    "SHAP SD":S.std(axis=0),
    "SHAP Min":S.min(axis=0),
    "SHAP Max":S.max(axis=0)
})
total=global_shap["Mean absolute SHAP"].sum()
global_shap["Relative importance (%)"]=100*global_shap["Mean absolute SHAP"]/total if total>0 else 0
global_shap=global_shap.sort_values("Mean absolute SHAP",ascending=False).reset_index(drop=True)

print("\nGLOBAL SHAP IMPORTANCE - CHE, CO2 AND EA")
print("Best algorithm:",best_name)
print("Number of observations:",len(X3))
print(global_shap.to_string(index=False,float_format=lambda v:f"{v:.6f}"))
display(global_shap.style.format({c:"{:.6f}" for c in global_shap.columns if c!="Variable"}).background_gradient(cmap="YlGnBu",subset=["Mean absolute SHAP","Relative importance (%)"]).set_properties(**{"color":"black","font-weight":"bold"}))

plt.rcParams.update({
    "font.family":"DejaVu Sans",
    "axes.labelcolor":"black",
    "xtick.color":"black",
    "ytick.color":"black",
    "axes.titleweight":"bold",
    "figure.facecolor":"white",
    "axes.facecolor":"white"
})
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(20,11),gridspec_kw={"width_ratios":[1.1,1.15]},facecolor="white")

p=global_shap.iloc[::-1]
palette={"CHE":"#4E79A7","CO2":"#F28E2B","EA":"#00A6A6"}
colors=[palette[v] for v in p["Variable"]]
bars=ax1.barh(p["Variable"],p["Mean absolute SHAP"],color=colors,edgecolor="black",linewidth=1.4,height=.72)

ax1.set_title("(a) Global SHAP Importance",fontsize=22,fontweight="bold",color="black",pad=22)
ax1.set_xlabel("Mean absolute SHAP value",fontsize=18,fontweight="bold",color="black",labelpad=14)
ax1.set_xlim(0,max(p["Mean absolute SHAP"].max()*1.7,1e-8))
ax1.set_ylim(-.8,len(p)-.2)
ax1.tick_params(axis="both",labelsize=17,colors="black",length=6,width=1.2)
for label in ax1.get_xticklabels()+ax1.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
for bar,val,pct in zip(bars,p["Mean absolute SHAP"],p["Relative importance (%)"]):
    ax1.text(val+ax1.get_xlim()[1]*.018,bar.get_y()+bar.get_height()/2,
             f"{val:.4f} ({pct:.2f}%)",va="center",ha="left",
             fontsize=17,fontweight="bold",color="black")
ax1.grid(axis="x",alpha=.18,linestyle="--",linewidth=1)
ax1.set_axisbelow(True)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["left"].set_linewidth(1.2)
ax1.spines["bottom"].set_linewidth(1.2)

plt.sca(ax2)
shap.summary_plot(S,X3,feature_names=features,max_display=3,show=False,plot_size=None,color_bar=False)
ax2=plt.gca()
ax2.set_title("(b) SHAP Beeswarm Summary",fontsize=22,fontweight="bold",color="black",pad=22)
ax2.set_xlabel("SHAP value (impact on predicted HDI)",fontsize=18,fontweight="bold",color="black",labelpad=14)
ax2.tick_params(axis="both",labelsize=17,colors="black",length=6,width=1.2)
for label in ax2.get_xticklabels()+ax2.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
ax2.axvline(0,color="black",linewidth=1.2,alpha=.7)
ax2.grid(axis="x",alpha=.15,linestyle="--")
ax2.set_axisbelow(True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

fig.suptitle(f"SHAP-Based HDI Model Interpretability: {best_name}",fontsize=25,fontweight="bold",color="black",y=.98)
fig.subplots_adjust(left=.10,right=.97,bottom=.13,top=.86,wspace=.43)
fig.savefig("Figure_SHAP_CHE_CO2_EA_Colored.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

global_shap.to_csv("Table_SHAP_CHE_CO2_EA.csv",index=False)
print("\nSHAP IMPORTANCE RANKING")
for i,row in global_shap.iterrows():
    print(f"{i+1}. {row['Variable']}: Mean |SHAP| = {row['Mean absolute SHAP']:.6f}, Relative importance = {row['Relative importance (%)']:.2f}%")


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

S=np.asarray(sv.values if hasattr(sv,"values") else sv)
F=list(features)
if S.ndim!=2 or S.shape[1]!=len(F):
    raise ValueError("SHAP matrisi ile özellik sayısı uyumsuz.")
if len(X)!=S.shape[0]:
    raise ValueError("X ile SHAP gözlem sayısı uyumsuz.")

global_shap=pd.DataFrame({
    "Variable":F,
    "Mean absolute SHAP":np.abs(S).mean(axis=0),
    "Mean signed SHAP":S.mean(axis=0),
    "SHAP SD":S.std(axis=0),
    "SHAP Min":S.min(axis=0),
    "SHAP Max":S.max(axis=0)
})
total=global_shap["Mean absolute SHAP"].sum()
global_shap["Relative importance (%)"]=100*global_shap["Mean absolute SHAP"]/total if total>0 else 0
global_shap=global_shap.sort_values("Mean absolute SHAP",ascending=False).reset_index(drop=True)

print("\nGLOBAL SHAP IMPORTANCE - FULL NUMERICAL RESULTS")
print(global_shap.to_string(index=False,float_format=lambda v:f"{v:.3f}"))
display(global_shap.style.format({c:"{:.6f}" for c in global_shap.columns if c!="Variable"}).background_gradient(cmap="YlGnBu",subset=["Mean absolute SHAP","Relative importance (%)"]).set_properties(**{"color":"black","font-weight":"bold"}))

plt.rcParams.update({"font.family":"DejaVu Sans","axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(20,12),gridspec_kw={"width_ratios":[1,1.15]},facecolor="white")
p=global_shap.iloc[::-1]
colors=plt.cm.Greens(np.linspace(.45,.85,len(p)))
bars=ax1.barh(p["Variable"],p["Mean absolute SHAP"],color=colors,edgecolor="black",linewidth=1.1,height=.65)
ax1.set_title("(a) Global SHAP Importance",fontsize=21,fontweight="bold",color="black",pad=18)
ax1.set_xlabel("Mean absolute SHAP value",fontsize=17,fontweight="bold",color="black")
ax1.set_xlim(0,max(p["Mean absolute SHAP"].max()*1.65,1e-8))
ax1.tick_params(axis="both",labelsize=16,colors="black")
for label in ax1.get_xticklabels()+ax1.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
for bar,val,pct in zip(bars,p["Mean absolute SHAP"],p["Relative importance (%)"]):
    ax1.text(val+ax1.get_xlim()[1]*.015,bar.get_y()+bar.get_height()/2,f"{val:.3f} ({pct:.3f}%)",va="center",fontsize=15,fontweight="bold",color="black")
ax1.grid(axis="x",alpha=.15,linestyle="--")
ax1.set_axisbelow(True)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

plt.sca(ax2)
shap.summary_plot(S,X,feature_names=F,max_display=len(F),show=False,plot_size=None,color_bar=False)
ax2=plt.gca()
ax2.set_title("(b) SHAP Beeswarm Summary",fontsize=21,fontweight="bold",color="black",pad=18)
ax2.set_xlabel("SHAP value (impact on predicted HDI)",fontsize=17,fontweight="bold",color="black")
ax2.tick_params(axis="both",labelsize=16,colors="black")
for label in ax2.get_xticklabels()+ax2.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
ax2.axvline(0,color="black",linewidth=1,alpha=.6)
ax2.grid(axis="x",alpha=.12,linestyle="--")
ax2.set_axisbelow(True)

fig.suptitle(f"SHAP-Based Model Interpretability: {best_name}",fontsize=24,fontweight="bold",color="black",y=.99)
fig.subplots_adjust(left=.12,right=.97,bottom=.14,top=.84,wspace=.45)
fig.savefig("Figure_06_07_Combined_SHAP_Analysis.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

print("\nSHAP IMPORTANCE RANKING")
for i,row in global_shap.iterrows():
    print(f"{i+1}. {row['Variable']}: Mean |SHAP| = {row['Mean absolute SHAP']:.3f}, Relative importance = {row['Relative importance (%)']:.2f}%")


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

S=np.asarray(sv.values if hasattr(sv,"values") else sv)
F=list(features)
if S.ndim!=2 or S.shape[1]!=len(F):
    raise ValueError("SHAP matrisi ile özellik sayısı uyumsuz.")
if len(X)!=S.shape[0]:
    raise ValueError("X ile SHAP gözlem sayısı uyumsuz.")

global_shap=pd.DataFrame({
    "Variable":F,
    "Mean absolute SHAP":np.abs(S).mean(axis=0),
    "Mean signed SHAP":S.mean(axis=0),
    "SHAP SD":S.std(axis=0),
    "SHAP Min":S.min(axis=0),
    "SHAP Max":S.max(axis=0)
})
total=global_shap["Mean absolute SHAP"].sum()
global_shap["Relative importance (%)"]=100*global_shap["Mean absolute SHAP"]/total if total>0 else 0
global_shap=global_shap.sort_values("Mean absolute SHAP",ascending=False).reset_index(drop=True)

print("\nGLOBAL SHAP IMPORTANCE - FULL NUMERICAL RESULTS")
print(global_shap.to_string(index=False,float_format=lambda v:f"{v:.3f}"))
display(global_shap.style.format({c:"{:.3f}" for c in global_shap.columns if c!="Variable"}).background_gradient(cmap="YlGnBu",subset=["Mean absolute SHAP","Relative importance (%)"]).set_properties(**{"color":"black","font-weight":"bold"}))

plt.rcParams.update({"font.family":"DejaVu Sans","axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(20,12),gridspec_kw={"width_ratios":[1,1.15]},facecolor="white")

p=global_shap.iloc[::-1]
vivid_colors=["#00A878","#2878B5","#F28E2B","#9B59B6","#E15759","#17BECF","#EDC948","#FF6F91","#4E79A7","#B07AA1"]
color_map=dict(zip(global_shap["Variable"],vivid_colors[:len(global_shap)])) if len(global_shap)<=len(vivid_colors) else dict(zip(global_shap["Variable"],plt.cm.tab20(np.linspace(0,1,len(global_shap)))))
bars=ax1.barh(p["Variable"],p["Mean absolute SHAP"],color=[color_map[v] for v in p["Variable"]],edgecolor="black",linewidth=1.2,height=.65)
ax1.set_title("(a) Global SHAP Importance",fontsize=21,fontweight="bold",color="black",pad=18)
ax1.set_xlabel("Mean absolute SHAP value",fontsize=17,fontweight="bold",color="black")
ax1.set_xlim(0,max(p["Mean absolute SHAP"].max()*1.65,1e-8))
ax1.tick_params(axis="both",labelsize=16,colors="black")
for label in ax1.get_xticklabels()+ax1.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
for bar,val,pct in zip(bars,p["Mean absolute SHAP"],p["Relative importance (%)"]):
    ax1.text(val+ax1.get_xlim()[1]*.015,bar.get_y()+bar.get_height()/2,f"{val:.3f} ({pct:.3f}%)",va="center",fontsize=15,fontweight="bold",color="black")
ax1.grid(axis="x",alpha=.15,linestyle="--")
ax1.set_axisbelow(True)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

plt.sca(ax2)
shap.summary_plot(S,X,feature_names=F,max_display=len(F),show=False,plot_size=None,color_bar=False)
ax2=plt.gca()
ax2.set_title("(b) SHAP Beeswarm Summary",fontsize=21,fontweight="bold",color="black",pad=18)
ax2.set_xlabel("SHAP value (impact on predicted HDI)",fontsize=17,fontweight="bold",color="black")
ax2.tick_params(axis="both",labelsize=16,colors="black")
for label in ax2.get_xticklabels()+ax2.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")
ax2.axvline(0,color="black",linewidth=1,alpha=.6)
ax2.grid(axis="x",alpha=.12,linestyle="--")
ax2.set_axisbelow(True)

fig.suptitle(f"SHAP-Based Model Interpretability: {best_name}",fontsize=24,fontweight="bold",color="black",y=.99)
fig.subplots_adjust(left=.12,right=.97,bottom=.14,top=.84,wspace=.45)
fig.savefig("Figure_06_07_Combined_SHAP_Analysis_Colorful.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()

print("\nSHAP IMPORTANCE RANKING")
for i,row in global_shap.iterrows():
    print(f"{i+1}. {row['Variable']}: Mean |SHAP| = {row['Mean absolute SHAP']:.3f}, Relative importance = {row['Relative importance (%)']:.2f}%")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import shap

S=np.asarray(sv.values if hasattr(sv,"values") else sv)
F=list(features)
if len(F)!=3:
    raise ValueError(f"Üç değişken bekleniyor; mevcut değişken sayısı: {len(F)}")
if S.shape!=(len(X),len(F)):
    raise ValueError("SHAP değerleri ile X boyutları uyumsuz.")

fig,axes=plt.subplots(1,3,figsize=(24,7),facecolor="white")
for i,(ax,feature) in enumerate(zip(axes,F)):
    plt.sca(ax)
    shap.dependence_plot(feature,S,X,feature_names=F,interaction_index="auto",ax=ax,show=False)
    ax.set_title(f"({chr(97+i)}) {feature}",fontsize=21,fontweight="bold",color="black",pad=15)
    ax.set_xlabel(feature,fontsize=18,fontweight="bold",color="black")
    ax.set_ylabel("SHAP value (HDI scale)",fontsize=18,fontweight="bold",color="black")
    ax.tick_params(axis="both",labelsize=16,colors="black")
    for label in ax.get_xticklabels()+ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")
    ax.axhline(0,color="black",linestyle="--",linewidth=1.2,alpha=.6)
    ax.grid(alpha=.15,linestyle="--")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(f"SHAP Dependence Analysis: {best_name}",fontsize=25,fontweight="bold",color="black",y=1.03)
fig.tight_layout(w_pad=3)
fig.savefig("Figure_08_SHAP_Dependence_Combined.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import shap

S=np.asarray(sv.values if hasattr(sv,"values") else sv)
F=list(features)
if len(F)!=3:
    raise ValueError(f"Üç değişken bekleniyor; mevcut değişken sayısı: {len(F)}")
if S.shape!=(len(X),len(F)):
    raise ValueError("SHAP değerleri ile X boyutları uyumsuz.")

fig,axes=plt.subplots(3,1,figsize=(12,21),facecolor="white")
for i,(ax,feature) in enumerate(zip(axes,F)):
    plt.sca(ax)
    shap.dependence_plot(feature,S,X,feature_names=F,interaction_index="auto",ax=ax,show=False)
    ax.set_title(f"({chr(97+i)}) {feature}",fontsize=22,fontweight="bold",color="black",pad=16)
    ax.set_xlabel(feature,fontsize=19,fontweight="bold",color="black")
    ax.set_ylabel("SHAP value (HDI scale)",fontsize=19,fontweight="bold",color="black")
    ax.tick_params(axis="both",labelsize=18,colors="black")
    for label in ax.get_xticklabels()+ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")
    ax.axhline(0,color="black",linestyle="--",linewidth=1.2,alpha=.6)
    ax.grid(alpha=.15,linestyle="--")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(f"SHAP Dependence Analysis: {best_name}",fontsize=25,fontweight="bold",color="black",y=.995)
fig.subplots_adjust(left=.13,right=.88,bottom=.06,top=.94,hspace=.45)
fig.savefig("Figure_08_SHAP_Dependence_3x1.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.show()


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED=42
features=["LE","EYS","MYS","GNI","CHE","CO2","EA"]
X=df[features].apply(pd.to_numeric,errors="raise").copy()
y=pd.to_numeric(df["HDI"],errors="raise")
assert X.shape==(191,7) and y.notna().all() and X.notna().all().all()

models={
"Dummy Mean":DummyRegressor(strategy="mean"),
"KNN Regression":Pipeline([("scaler",StandardScaler()),("model",KNeighborsRegressor(n_neighbors=5,weights="distance",p=2))]),
"Elastic Net":Pipeline([("scaler",StandardScaler()),("model",ElasticNet(alpha=.01,l1_ratio=.5,max_iter=20000))]),
"SVR RBF":Pipeline([("scaler",StandardScaler()),("model",SVR(kernel="rbf",C=10,epsilon=.01,gamma="scale"))]),
"Random Forest":RandomForestRegressor(n_estimators=500,max_features=1.,min_samples_leaf=2,random_state=SEED,n_jobs=-1),
"Extra Trees":ExtraTreesRegressor(n_estimators=500,max_features=1.,min_samples_leaf=2,random_state=SEED,n_jobs=-1),
"Gradient Boosting":GradientBoostingRegressor(n_estimators=300,learning_rate=.03,max_depth=2,loss="huber",random_state=SEED),
"Hist Gradient Boosting":HistGradientBoostingRegressor(max_iter=300,learning_rate=.04,max_leaf_nodes=7,l2_regularization=2,random_state=SEED),
"CatBoost":CatBoostRegressor(iterations=500,depth=4,learning_rate=.03,loss_function="RMSE",l2_leaf_reg=5,random_seed=SEED,verbose=False,allow_writing_files=False),
"LightGBM":LGBMRegressor(n_estimators=500,max_depth=4,num_leaves=7,learning_rate=.03,min_child_samples=5,reg_lambda=5,random_state=SEED,n_jobs=-1,verbosity=-1),
"XGBoost":XGBRegressor(n_estimators=500,max_depth=3,learning_rate=.03,subsample=.85,colsample_bytree=.85,reg_lambda=5,objective="reg:squarederror",random_state=SEED,n_jobs=-1)
}

cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
oof_predictions={}
rows=[]
for name,model in models.items():
    pred=cross_val_predict(clone(model),X,y,cv=cv,n_jobs=1)
    oof_predictions[name]=pred
    rows.append({"Algorithm":name,"OOF_R2":r2_score(y,pred),"OOF_RMSE":np.sqrt(mean_squared_error(y,pred)),"OOF_MAE":mean_absolute_error(y,pred)})
results=pd.DataFrame(rows).sort_values("OOF_RMSE").reset_index(drop=True)
best_name=results.loc[results["Algorithm"]!="Dummy Mean","Algorithm"].iloc[0]
best_model=clone(models[best_name]).fit(X,y)

params=pd.DataFrame([{"Algorithm":name,"Hyperparameter configuration":str(model.get_params(deep=False))} for name,model in models.items()])
with pd.ExcelWriter("Table_3_ML_Algorithms_and_Performance.xlsx") as writer:
    params.to_excel(writer,sheet_name="Parameters",index=False)
    results.to_excel(writer,sheet_name="Performance",index=False)

plt.rcParams.update({"font.family":"DejaVu Sans","font.size":12,"axes.labelweight":"bold","axes.titleweight":"bold","savefig.facecolor":"white"})

fig,axes=plt.subplots(1,3,figsize=(19,7),facecolor="white")
for ax,(metric,title,ascending) in zip(axes,[("OOF_R2","R²",False),("OOF_RMSE","RMSE",True),("OOF_MAE","MAE",True)]):
    p=results.sort_values(metric,ascending=ascending)
    bars=ax.barh(p["Algorithm"],p[metric],color="#159A72",edgecolor="black")
    ax.invert_yaxis()
    ax.set_title(title,fontsize=16)
    ax.grid(axis="x",alpha=.15)
    ax.set_axisbelow(True)
    ax.set_xlim(min(0,p[metric].min()),p[metric].max()*1.22)
    for b,v in zip(bars,p[metric]):
        ax.text(v+p[metric].max()*.015,b.get_y()+b.get_height()/2,f"{v:.3f}",va="center",fontsize=10,fontweight="bold")
fig.tight_layout()
fig.savefig("Figure_14_ML_Performance.png",dpi=600,bbox_inches="tight")
plt.show()

observed=y.to_numpy()
pred=np.asarray(oof_predictions[best_name])
fig,ax=plt.subplots(figsize=(8,7),facecolor="white")
ax.scatter(observed,pred,s=55,c="#159A72",edgecolors="black",alpha=.85)
lim=[min(observed.min(),pred.min())-.03,max(observed.max(),pred.max())+.03]
ax.plot(lim,lim,"--",color="black",linewidth=2)
ax.set(xlim=lim,ylim=lim,xlabel="Observed HDI",ylabel="Out-of-fold Predicted HDI")
ax.set_title(f"{best_name}: R² = {r2_score(observed,pred):.4f}, RMSE = {np.sqrt(mean_squared_error(observed,pred)):.4f}")
ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig("Figure_15_Observed_vs_Predicted_HDI.png",dpi=600,bbox_inches="tight")
plt.show()

background=X.sample(n=min(50,len(X)),random_state=SEED)
explainer=shap.PermutationExplainer(best_model.predict,shap.maskers.Independent(background),seed=SEED)
sv=explainer(X,max_evals=2*X.shape[1]+1)
S=np.asarray(sv.values)
assert S.shape==X.shape

global_shap=pd.DataFrame({"Variable":features,"Mean absolute SHAP":np.abs(S).mean(axis=0)})
global_shap["Relative importance (%)"]=100*global_shap["Mean absolute SHAP"]/global_shap["Mean absolute SHAP"].sum()
global_shap=global_shap.sort_values("Mean absolute SHAP",ascending=False)
print("\nMODEL PERFORMANCE\n",results.to_string(index=False))
print("\nGLOBAL SHAP IMPORTANCE\n",global_shap.to_string(index=False))

fig,axes=plt.subplots(1,2,figsize=(17,7),facecolor="white")
p=global_shap.iloc[::-1]
axes[0].barh(p["Variable"],p["Mean absolute SHAP"],color="#159A72",edgecolor="black")
axes[0].set_title("(a) Global SHAP Importance")
axes[0].set_xlabel("Mean absolute SHAP value")
axes[0].set_xlim(0,p["Mean absolute SHAP"].max()*1.55)
for i,(_,r) in enumerate(p.iterrows()):
    axes[0].text(r["Mean absolute SHAP"]+.001,i,f'{r["Relative importance (%)"]:.2f}%',va="center",fontweight="bold")
plt.sca(axes[1])
shap.summary_plot(S,X,show=False,plot_size=None,max_display=7)
axes[1].set_title("(b) SHAP Beeswarm")
fig.tight_layout()
fig.savefig("Figure_16_Global_SHAP_and_Beeswarm.png",dpi=600,bbox_inches="tight")
plt.show()

top=global_shap["Variable"].head(3).tolist()
fig,axes=plt.subplots(1,3,figsize=(18,5),facecolor="white")
for ax,f in zip(axes,top):
    j=features.index(f)
    ax.scatter(X[f],S[:,j],s=38,color="#159A72",edgecolors="black",linewidths=.3,alpha=.8)
    ax.axhline(0,color="black",linestyle="--",linewidth=1)
    ax.set(xlabel=f,ylabel="SHAP value (HDI scale)",title=f)
    ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig("Figure_17_SHAP_Dependence.png",dpi=600,bbox_inches="tight")
plt.show()

In [ ]:
direction=pd.DataFrame({"Variable":features,"Mean SHAP":sv.mean(axis=0),"Median SHAP":np.median(sv,axis=0),"Positive contribution (%)":100*(sv>0).mean(axis=0),"Negative contribution (%)":100*(sv<0).mean(axis=0)})
show_table(direction,"DIRECTION OF SHAP CONTRIBUTIONS")
fig,ax=plt.subplots(figsize=(9,5))
xpos=np.arange(len(features))
width=0.36
b1=ax.bar(xpos-width/2,direction["Positive contribution (%)"],width,label="Positive",color="#00A878",edgecolor="black")
b2=ax.bar(xpos+width/2,direction["Negative contribution (%)"],width,label="Negative",color="#D64F4F",edgecolor="black")
ax.set_xticks(xpos,features,fontweight="bold")
ax.set_ylabel("Share of countries (%)",fontweight="bold")
ax.set_title("Positive and Negative SHAP Contributions",fontweight="bold")
ax.legend()
for bars in [b1,b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f"{bar.get_height():.1f}%",ha="center",fontweight="bold",color="black")
ax.set_ylim(0,115)
savefig("Figure_09_SHAP_Direction")

In [ ]:
rng=np.random.default_rng(SEED)
boot_imp=np.zeros((B,len(features)))
for b in range(B):
    idx=rng.integers(0,len(X),len(X))
    boot_imp[b]=np.abs(sv[idx]).mean(axis=0)
boot_summary=pd.DataFrame({"Variable":features,"Mean absolute SHAP":np.abs(sv).mean(axis=0),"Bootstrap SE":boot_imp.std(axis=0,ddof=1),"CI 2.5%":np.percentile(boot_imp,2.5,axis=0),"CI 97.5%":np.percentile(boot_imp,97.5,axis=0)})
boot_summary=boot_summary.sort_values("Mean absolute SHAP",ascending=False).reset_index(drop=True)
show_table(boot_summary,"BOOTSTRAP SHAP IMPORTANCE AND 95% INTERVALS")
fig,ax=plt.subplots(figsize=(9,5))
p=boot_summary.sort_values("Mean absolute SHAP")
xerr=np.vstack([p["Mean absolute SHAP"]-p["CI 2.5%"],p["CI 97.5%"]-p["Mean absolute SHAP"]])
ax.barh(p["Variable"],p["Mean absolute SHAP"],xerr=xerr,color="#18A572",edgecolor="black",capsize=5,error_kw={"ecolor":"black","elinewidth":1.5})
ax.set_title("Bootstrap 95% Confidence Intervals of SHAP Importance",fontweight="bold")
ax.set_xlabel("Mean absolute SHAP value",fontweight="bold")
for i,val in enumerate(p["Mean absolute SHAP"]):
    ax.text(val,i,f" {val:.5f}",va="center",fontweight="bold",color="black")
savefig("Figure_10_SHAP_Bootstrap_CI")

In [ ]:
import numpy as np,pandas as pd

# 1. Model tahminleri ve artıklar
best_oof=np.asarray(oof_predictions[best_name]).ravel()
residuals=np.asarray(y).ravel()-best_oof
fitted_hdi=np.asarray(best_model.predict(X)).ravel()

# 2. SHAP temel değerleri ve yeniden oluşturulan tahminler
base_values=np.asarray(shap_exp.base_values).reshape(-1)
if len(base_values)==1:
    base_values=np.repeat(base_values,len(X))

shap_reconstruction=base_values+np.sum(sv,axis=1)

# 3. SHAP boyut ve gözlem kontrolleri
if sv.shape!=(len(X),X.shape[1]):
    raise ValueError(f"SHAP boyutu hatalı: {sv.shape}")

if not (len(data)==len(X)==len(y)==len(best_oof)==len(fitted_hdi)==len(base_values)):
    raise ValueError("Ülke, gözlem ve tahmin sayıları eşleşmiyor.")

# 4. Ülke bazında SHAP sonuçları
country_shap=pd.DataFrame({
    "Country":data["country"].to_numpy(),
    "Observed_HDI":np.asarray(y).ravel(),
    "OOF_Predicted_HDI":best_oof,
    "OOF_Residual":residuals,
    "Absolute_OOF_Error":np.abs(residuals),
    "Fitted_HDI":fitted_hdi,
    "SHAP_Base_Value":base_values,
    "SHAP_Reconstructed_HDI":shap_reconstruction
})

for j,feature in enumerate(X.columns):
    country_shap[f"{feature}_Value"]=X[feature].to_numpy()
    country_shap[f"SHAP_{feature}"]=sv[:,j]

country_shap["Total_Absolute_SHAP"]=np.abs(sv).sum(axis=1)
country_shap["SHAP_Reconstruction_Error"]=fitted_hdi-shap_reconstruction

# 5. Sonuç tabloları
print("SEÇİLEN MODEL:",best_name)
print("OOF RMSE:",round(np.sqrt(np.mean(residuals**2)),6))
print("SHAP maksimum yeniden oluşturma hatası:",round(np.max(np.abs(country_shap["SHAP_Reconstruction_Error"])),8))

show_table(
    country_shap.nlargest(10,"Total_Absolute_SHAP"),
    "COUNTRIES WITH THE LARGEST TOTAL SHAP CONTRIBUTIONS"
)

show_table(
    country_shap.nlargest(5,"Absolute_OOF_Error"),
    "COUNTRIES WITH THE LARGEST OOF PREDICTION ERRORS"
)

display(country_shap.round(5))